# Final QReCC Out-of-Domain Retrieval Evaluation

This notebook transfers the locked ROCC systems to the complete QReCC test split without further selection, evaluating out-of-domain generalization with BM25 and ANCE across IterCQR baselines, ROCC retrieval routes, fixed RRF fusion, and context budgets.

The code names `pretrained`, `imitation_control_e6`, and `bm25_treatment_e6` correspond to the thesis checkpoints $f_{\mathrm{base}}$, $f_{\mathrm{imit}}$, and $f_{\mathrm{BM25}}$. The base student has three training epochs. Both post-trained students start from that base and receive six additional epochs.

At budget $B$, `I`, `R`, and `D` denote $I_B$, $R_B$, and $D_B$. `I+R+D` denotes $IRD_B$, not an arithmetic sum of scores. Their rankings are $L_I^B$, $L_R^B$, $L_D^B$, and $L_{IRD}^B$. `RRF10` means reciprocal rank fusion with $\kappa=10$. `IterCQR-G+2S-RRF10` is the thesis comparison $G+2S$.

### ANCE session control

The QReCC ANCE index is evaluated in two disjoint GPU sessions because the complete index does not fit into one GPU run. `session_1` processes the first shard range and `session_2` the second. Both sessions use the same frozen query bundle and are merged only after both ranking stores are complete.

`NB07B_BACKEND=None` selects ANCE in Colab and BM25 locally. Fresh local `"ance"` mode uses the same two-session protocol and requires the prepared index plus sufficient GPU memory.

In [1]:
# Change between the two ANCE GPU runs (local or Colab).
# Both sessions must share the same query bundle and result directory.
QRECC_ANCE_SESSION_ID = "session_2"  # Use "session_1" first, then "session_2".
NB07B_BACKEND = "ance"  # None selects automatically; "ance" and "bm25" select explicitly.

### Evaluation setup

The evaluation supports local BM25 and GPU-based ANCE execution, locally or in Colab. Prepare the checkout and dependencies first; `ROCC_ROOT` may point to a local or mounted Drive checkout. Large checkpoints, indices, caches and results must persist across the two ANCE sessions. The existing artifact-root settings can point to persistent storage; no private repository or Azure credentials are required for local indices.


In [2]:
from __future__ import annotations

import gc
import gzip
import hashlib
import importlib.metadata
import inspect
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping, Sequence


# Use the already prepared checkout (ROCC_ROOT may point to a mounted Drive checkout).
def is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
    except ModuleNotFoundError:
        return False
    return True

IN_COLAB = is_colab_runtime()


def resolve_project_root(start: Path) -> Path:
    configured = os.environ.get("ROCC_ROOT") or os.environ.get("MASTER_THESIS_PROJECT_ROOT")
    if configured:
        candidate = Path(configured).expanduser().resolve()
        if not (candidate / "experiments" / "rocc").is_dir():
            raise FileNotFoundError(f"Configured ROCC checkout is missing: {candidate}")
        return candidate
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "rocc").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from your ROCC checkout, or set ROCC_ROOT to it. "
        "In Colab, first clone the public repository over HTTPS or authenticate "
        "your own private checkout; no private clone or deploy key is assumed."
    )


PROJECT_ROOT = resolve_project_root(Path.cwd())
os.environ["MASTER_THESIS_PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from experiments.rocc import (
    CachedIterCQRBM25Config,
    CachedIterCQRBM25Pipeline,
    HistorySelectorConfig,
    QRECC_TOTAL_PASSAGES,
    QueryRankingStore,
    SelectorExperimentConfig,
    check_dense_gpu_capacity,
    ensure_colab_retrieval_dependencies,
    evaluate_fixed_rrf_views,
    inspect_bm25_lucene_index,
    load_bm25_retriever,
    load_crf_inference_model,
    load_dense_ance_torch_retriever,
    load_itercqr_rewriter,
    load_itercqr_tokenizer,
    load_or_compute_current_query_only,
    load_qrecc_frame,
    materialize_history_arm_inputs,
    materialize_selector_direct_queries,
    merge_query_ranking_stores,
    normalize_rewrite,
    paired_bootstrap_mean_ci,
    prepare_colab_itercqr_checkpoint,
    predict_crf_history_candidates,
    qrecc_ance_session_shard_range,
    qrecc_turn_selection_samples,
    read_selected_histories_jsonl,
    resolve_qrecc_resources,
    run_query_retrieval_to_store,
    serialize_itercqr_input,
    summarize_dense_faiss_shards,
    validate_itercqr_checkpoint,
    write_selected_histories_jsonl,
)


SEED = 13
BUDGETS = (64, 128, 256, 512)
PRIMARY_BUDGET = 64
D_NO_HISTORY_QUERY_RULE = "dataset_current_query_without_t5"
RRF_K = 10
RETRIEVAL_TOP_K = 1_000
EVAL_KS = (3, 10, 100, 1000)
BOOTSTRAP_REPLICATES = 10_000
ITER_SAMPLING_SEED = 13
ITER_SAMPLING_TEMPERATURE = 1.0
ITER_SAMPLING_TOP_K = 50
ITER_SAMPLING_TOP_P = 1.0
NUM_ITER_SAMPLES = 2
SELECTOR_BATCH_SIZE = 32
REWRITE_BATCH_SIZE = 64 if IN_COLAB else 16
RETRIEVAL_BATCH_SIZE = 64
RETRIEVAL_STORE_CHUNK_SIZE = 256
SAMPLING_SEED_SCHEME = "sha256_first_8_bytes_big_endian_v1"
SYSTEM_ORDER = (
    "pretrained",
    "imitation_control_e6",
    "bm25_treatment_e6",
)
METRIC_ORDER = ("MRR", "nDCG@3", "R@10", "R@100", "R@1000")

DEFAULT_BACKEND = "ance" if IN_COLAB else "bm25"
BACKEND = os.environ.get(
    "ROCC_NB07B_BACKEND",
    str(globals().get("NB07B_BACKEND") or DEFAULT_BACKEND),
).strip().lower()
if BACKEND not in {"bm25", "ance"}:
    raise ValueError("ROCC_NB07B_BACKEND must be bm25 or ance.")
QRECC_ANCE_SESSION_ID = os.environ.get(
    "ROCC_QRECC_ANCE_SESSION",
    str(globals().get("QRECC_ANCE_SESSION_ID", "session_1")),
).strip()
if QRECC_ANCE_SESSION_ID not in {"session_1", "session_2"}:
    raise ValueError("ROCC_QRECC_ANCE_SESSION must be session_1 or session_2.")

MODEL_ARTIFACT_ROOT = Path(
    os.environ.get(
        "ROCC_NB07B_MODEL_ARTIFACT_ROOT",
        str(PROJECT_ROOT),
    )
).expanduser().resolve()
RESULT_ARTIFACT_ROOT = Path(
    os.environ.get(
        "ROCC_NB07B_RESULT_ARTIFACT_ROOT",
        str(PROJECT_ROOT),
    )
).expanduser().resolve()
RESULT_DIR = (
    RESULT_ARTIFACT_ROOT
    / "experiments/results/07b_qrecc_ood"
)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
QUERY_BUNDLE_PATH = RESULT_DIR / "query_bundle/query_bundle.jsonl.gz"
QUERY_BUNDLE_MANIFEST_PATH = RESULT_DIR / "query_bundle/manifest.json"
BUNDLE_ARTIFACTS_PRESENT = (
    QUERY_BUNDLE_PATH.exists()
    or QUERY_BUNDLE_MANIFEST_PATH.exists()
)

DATA_DIR = PROJECT_ROOT / "experiments/data/qrecc"
ITERCQR_MODEL_DIR = (
    PROJECT_ROOT / "experiments/model/IterCQR/IterCQR Model"
)
BM25_INDEX_DIR = (
    PROJECT_ROOT
    / "experiments/data/pyserini_bm25_lucene_qrecc/lucene_index"
)
BM25_LOG_FILE = PROJECT_ROOT / "experiments/logs/qrecc_bm25_lucene_index.log"
DENSE_INDEX_DIR = (
    PROJECT_ROOT
    / "experiments/data/pyserini_ance_faiss_qrecc"
    / "faiss_flat_index_full_sharded"
)
# Indices and processed splits are prepared locally (or linked from Drive).

PRETRAINED_CHECKPOINT = (
    MODEL_ARTIFACT_ROOT
    / "experiments/results/05_history_selector/full_candidates"
    / "collapsed_crf/model.pt"
)
CONTROL_CHECKPOINT = (
    MODEL_ARTIFACT_ROOT
    / "experiments/results/06_post_training"
    / "controlled_hard_relabel_nostop_e6/training"
    / "imitation_control/checkpoint_epoch_6.pt"
)
TREATMENT_CHECKPOINT = (
    MODEL_ARTIFACT_ROOT
    / "experiments/results/06_post_training"
    / "controlled_hard_relabel_nostop_e6/training"
    / "bm25_treatment/checkpoint_epoch_6.pt"
)
CHECKPOINTS = {
    "pretrained": PRETRAINED_CHECKPOINT,
    "imitation_control_e6": CONTROL_CHECKPOINT,
    "bm25_treatment_e6": TREATMENT_CHECKPOINT,
}
ARCHITECTURE_DECISION_PATH = (
    MODEL_ARTIFACT_ROOT
    / "experiments/results/05_history_selector/train600"
    / "architecture/decision.json"
)
FUSION_DECISION_PATH = (
    MODEL_ARTIFACT_ROOT
    / "experiments/results/05_history_selector/train600"
    / "fusion/decision.json"
)
EXPECTED_CHECKPOINT_HASHES = {
    "pretrained": (
        "a07271abefbc37ef08dba4eb94adb0e165f53a652060c069c909a7a1614115c4"
    ),
    "imitation_control_e6": (
        "c754a91ac71111b82ca067d69490f29e7e6295fc368e4e5a3075e8866480dca6"
    ),
    "bm25_treatment_e6": (
        "a8d00d1cdcc7116e4f20833429a1acb879b2346bea4840319b555c073410992d"
    ),
}
EXPECTED_POPULATION_SHA256 = (
    "c9a19e8d6f7d6cf3ab1db89baf5a2d76da7f63e5e87d0a8f4dfd83f7b5c4fb1f"
)
EXPECTED_ITERCQR_MODEL_SHA256 = (
    "1a25ceaf597bf92ff2897cccb22e9c46b807efbf78a3234186066e1f62f16ae3"
)
EXPECTED_SERIALIZER_SHA256 = (
    "49cf2cd0dd6886f6513af55d007076fba1596374c5c1806c1ce6e8ca80414ae1"
)
POST_TRAINING_RECIPE_VERSION = (
    "hard_relabel_nltk_nostop_controlled_ce_crf_v2"
)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if IN_COLAB and not BUNDLE_ARTIFACTS_PRESENT:
    checkpoint_runtime = prepare_colab_itercqr_checkpoint(
        local_checkpoint=ITERCQR_MODEL_DIR,
    )
    display(pd.DataFrame([checkpoint_runtime.to_dict()]))
    if not checkpoint_runtime.complete:
        raise RuntimeError(
            "The complete IterCQR checkpoint was not found in Colab."
        )
    ITERCQR_MODEL_DIR = checkpoint_runtime.local_checkpoint

SELECTOR_CONFIG = HistorySelectorConfig(
    model_name="sentence-transformers/all-MiniLM-L12-v2",
    max_length=512,
    history_order="recent_first",
)
PIPELINE_CONFIG = CachedIterCQRBM25Config(
    cache_db=RESULT_DIR / "cache/rewrite_cache.sqlite3",
    model_dir=ITERCQR_MODEL_DIR,
    bm25_index_dir=BM25_INDEX_DIR,
    device=str(DEVICE),
    rewrite_batch_size=REWRITE_BATCH_SIZE,
    retrieval_batch_size=RETRIEVAL_BATCH_SIZE,
    retrieval_workers=max(1, min(8, (os.cpu_count() or 2) // 2)),
    retrieval_top_k=RETRIEVAL_TOP_K,
    eval_ks=EVAL_KS,
    k1=0.82,
    b=0.68,
)
EXPERIMENT_CONFIG = SelectorExperimentConfig(
    train_labels=(
        MODEL_ARTIFACT_ROOT
        / "experiments/results/04_teacher"
        / "rocc_history_labels_topiocqa_train.jsonl"
    ),
    train_manifest=(
        MODEL_ARTIFACT_ROOT
        / "experiments/results/04_teacher"
        / "rocc_history_labels_topiocqa_train.manifest.json"
    ),
    topiocqa_data_dir=DATA_DIR,
    result_dir=RESULT_DIR,
    encoding_cache_dir=RESULT_DIR / "cache",
    itercqr_model_dir=ITERCQR_MODEL_DIR,
    bm25_index_dir=BM25_INDEX_DIR,
    pipeline_cache_db=PIPELINE_CONFIG.cache_db,
    retrieval_top_k=RETRIEVAL_TOP_K,
    eval_ks=EVAL_KS,
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def canonical_json(value: Any) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def write_frame(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    frame.to_csv(
        temporary,
        index=False,
        float_format="%.17g",
        lineterminator="\n",
    )
    temporary.replace(path)


def write_bundle(path: Path, rows: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    with gzip.open(temporary, "wt", encoding="utf-8") as handle:
        for row in rows.to_dict(orient="records"):
            handle.write(canonical_json(row) + "\n")
    temporary.replace(path)


def read_bundle(path: Path) -> pd.DataFrame:
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        return pd.DataFrame(
            json.loads(line) for line in handle if line.strip()
        )


def history_word_count(history: Sequence[Mapping[str, Any]]) -> int:
    return sum(
        len(re.findall(r"\S+", str(turn.get(field, ""))))
        for turn in history
        for field in ("question", "answer")
    )


display(
    pd.DataFrame(
        [
            {
                "in_colab": IN_COLAB,
                "backend": BACKEND,
                "ance_session": (
                    QRECC_ANCE_SESSION_ID if BACKEND == "ance" else None
                ),
                "device": str(DEVICE),
                "result_dir": str(RESULT_DIR),
                "model_artifact_root": str(MODEL_ARTIFACT_ROOT),
            }
        ]
    )
)


,in_colab,backend,ance_session,device,result_dir,model_artifact_root
0,False,ance,session_2,cuda:0,/experiments/...,


### QReCC resources and evaluation population

NB07b restores the processed QReCC splits when required so that all gold passages use the numeric IDs of the fixed 54.57-million-passage collection. The complete test split is loaded before filtering because turns without gold passages may still provide history for later evaluated queries.

The established QReCC adapter reconstructs each alternating `Context` sequence into question-and-answer turns. Evaluation is then restricted to the 8,209 queries with at least one mapped gold passage. This preserves complete conversational histories while ensuring that every scored query has a valid retrieval target.

In [3]:
resources = resolve_qrecc_resources(DATA_DIR)
processed_ready = all(
    [
        resources.processed_train_json.exists(),
        resources.processed_test_json.exists(),
        resources.collection_manifest_json.exists(),
    ]
)
if not processed_ready:
    raise FileNotFoundError(
        f"QReCC processed splits/collection manifest are missing under {DATA_DIR}. "
        "Run notebook 00a first, or restore its matching processed and collection files. "
        "No private Azure account is required."
    )

all_frame = load_qrecc_frame(
    resources,
    prefer_processed=True,
    gold_only=False,
)
test_frame = all_frame.loc[
    all_frame["split"].eq("test")
].copy()
test_frame["sample_id"] = [
    f"test:{conv_id}:{turn_id}"
    for conv_id, turn_id in zip(
        test_frame["conv_id"],
        test_frame["turn_id"],
        strict=True,
    )
]
turn_samples = qrecc_turn_selection_samples(test_frame)
turn_sample_by_id = {
    str(sample.sample_id): sample for sample in turn_samples
}

gold_frame = test_frame.loc[
    test_frame["positive_ctx_passage_ids"].apply(bool)
].sort_values(["conv_id", "turn_id"], kind="mergesort")
sample_ids = gold_frame["sample_id"].astype(str).tolist()
if len(sample_ids) != 8_209 or len(sample_ids) != len(set(sample_ids)):
    raise RuntimeError(
        f"Unexpected QReCC evaluation population: {len(sample_ids)}"
    )
if not set(sample_ids).issubset(turn_sample_by_id):
    raise RuntimeError("QReCC gold queries are missing from the turn samples.")


@dataclass(frozen=True)
class QReCCConversationSample:
    sample_id: str
    split: str
    conv_id: int
    turn_id: int
    current_query: str
    history: list[dict[str, Any]]


def sample_history(sample: Any) -> list[dict[str, Any]]:
    return [
        {
            "turn_id": index,
            "question": str(unit.question),
            "answer": str(unit.answer),
        }
        for index, unit in enumerate(
            sample.history_units,
            start=1,
        )
    ]


samples = []
for sample_id in sample_ids:
    source = turn_sample_by_id[sample_id]
    samples.append(
        QReCCConversationSample(
            sample_id=sample_id,
            split="test",
            conv_id=int(source.conv_id),
            turn_id=int(source.turn_id),
            current_query=str(source.question),
            history=sample_history(source),
        )
    )
sample_by_id = {sample.sample_id: sample for sample in samples}
depth_by_sample = {
    sample.sample_id: len(sample.history) for sample in samples
}
source_by_sample = dict(
    zip(
        gold_frame["sample_id"].astype(str),
        gold_frame["conversation_source"].astype(str),
        strict=True,
    )
)
gold_by_sample = {
    str(row.sample_id): [
        str(docid) for docid in row.positive_ctx_passage_ids
    ]
    for row in gold_frame.itertuples(index=False)
}
population_sha256 = sha256_text("\n".join(sorted(sample_ids)))
if population_sha256 != EXPECTED_POPULATION_SHA256:
    raise RuntimeError(
        f"QReCC population hash has drifted: {population_sha256}"
    )

depth_counts = pd.Series(depth_by_sample).value_counts().sort_index()
population_manifest = {
    "schema_version": 1,
    "dataset": "QReCC test",
    "raw_test_rows": int(len(test_frame)),
    "evaluated_queries": int(len(sample_ids)),
    "sample_id_sha256": population_sha256,
    "history_depth_counts": {
        str(int(depth)): int(count)
        for depth, count in depth_counts.items()
    },
    "source_counts": {
        str(source): int(count)
        for source, count in gold_frame[
            "conversation_source"
        ].value_counts().sort_index().items()
    },
}
write_json(RESULT_DIR / "protocol/population.json", population_manifest)
display(pd.DataFrame([population_manifest]))
display(
    depth_counts.rename_axis("history_depth").rename("queries").reset_index()
)


,schema_version,dataset,raw_test_rows,evaluated_queries,sample_id_sha256,history_depth_counts,source_counts
0,1,QReCC test,16451,8209,c9a19e8d6f7d6cf3ab1db89baf5a2d76da7f63e5e87d0a...,"{'0': 1357, '1': 1343, '2': 1280, '3': 1245, '...","{'nq': 1442, 'quac': 6396, 'trec': 371}"


,history_depth,queries
0,0,1357
1,1,1343
2,2,1280
3,3,1245
4,4,1100
5,5,629
6,6,525
7,7,339
8,8,208
9,9,112


The final evaluation contains 8,209 queries with mapped gold passages. Earlier test turns remain available as conversational history even when they are not scored.

History selection is relevant for most queries. 83.5% contain prior context, 67.1% contain at least two history turns, and depths extend to 11 turns. The aggregate result is primarily shaped by QuAC, which contributes 77.9% of the population, followed by NQ with 17.6% and TREC with 4.5%. The population hash fixes the same 8,209 queries across BM25 and ANCE.

### Frozen evaluation queries

BM25 and ANCE run in different environments, while ANCE is distributed across two sessions. Regenerating queries within each retrieval run could introduce variation from model execution, caching, serialization, or stochastic sampling. The frozen bundle makes the query text a controlled input, so retrieval differences can be attributed to the backend.

The bundle also separates the experimental questions. `I`, `R`, and `D` reveal whether gains originate from rewriting history truncated to the input budget, rewriting ROCC-selected history, or direct retrieval from the budgeted selected-history input. The budget sweep measures sensitivity to available context. The B64 control and treatment routes test whether post-training transfers to QReCC. IterCQR greedy decoding and two fixed samples provide the corresponding non-ROCC baselines.

In [4]:
# Only explicit local mode binds newly trained checkpoints to their manifests.
from experiments.rocc.paths import bind_notebook_lineage
EXPECTED_CHECKPOINT_HASHES = bind_notebook_lineage(
    RESULT_DIR, "checkpoint_inputs", EXPECTED_CHECKPOINT_HASHES, CHECKPOINTS,
    manifest_fields=(
        ("pretrained", PRETRAINED_CHECKPOINT.parent / "manifest.json", ("checkpoint_sha256",)),
        ("pretrained", ARCHITECTURE_DECISION_PATH, ("selector_checkpoint_sha256",)),
        ("imitation_control_e6", CONTROL_CHECKPOINT.parent / "manifest.json", ("epoch_checkpoints", "6", "sha256")),
        ("bm25_treatment_e6", TREATMENT_CHECKPOINT.parent / "manifest.json", ("epoch_checkpoints", "6", "sha256")),
    ),
)

BUNDLE_PROTOCOL = "nb07b_qrecc_ood_query_bundle_v3"
BUNDLE_COLUMNS = (
    "artifact_id",
    "family",
    "system",
    "sample_id",
    "conversation_source",
    "history_depth",
    "budget",
    "view",
    "candidate_index",
    "query_id",
    "query",
    "query_norm",
    "input_key",
    "input_length",
    "raw_input_length",
    "was_truncated",
    "original_history_words",
    "selected_history_words",
)


def expected_bundle_identity() -> dict[str, Any]:
    return {
        "protocol": BUNDLE_PROTOCOL,
        "population_sha256": population_sha256,
        "checkpoint_sha256": EXPECTED_CHECKPOINT_HASHES,
        "itercqr_model_sha256": EXPECTED_ITERCQR_MODEL_SHA256,
        "serializer_sha256": EXPECTED_SERIALIZER_SHA256,
        "budgets": list(BUDGETS),
        "primary_budget": PRIMARY_BUDGET,
        "d_no_history_query_rule": D_NO_HISTORY_QUERY_RULE,
        "rrf_k": RRF_K,
        "iter_sampling_seed": ITER_SAMPLING_SEED,
        "iter_sampling_temperature": ITER_SAMPLING_TEMPERATURE,
        "iter_sampling_top_k": ITER_SAMPLING_TOP_K,
        "iter_sampling_top_p": ITER_SAMPLING_TOP_P,
        "iter_samples": NUM_ITER_SAMPLES,
        "sampling_seed_scheme": SAMPLING_SEED_SCHEME,
        "expected_rows": len(sample_ids) * 30,
    }


def bundle_is_valid() -> bool:
    if not (
        QUERY_BUNDLE_PATH.is_file()
        and QUERY_BUNDLE_MANIFEST_PATH.is_file()
    ):
        return False
    manifest = json.loads(
        QUERY_BUNDLE_MANIFEST_PATH.read_text(encoding="utf-8")
    )
    expected = expected_bundle_identity()
    return bool(
        manifest.get("complete") is True
        and all(manifest.get(key) == value for key, value in expected.items())
        and manifest.get("latency_measured") is False
        and manifest.get("query_bundle_sha256")
        == sha256_file(QUERY_BUNDLE_PATH)
    )


def validate_generation_artifacts() -> dict[str, Any]:
    for system, checkpoint in CHECKPOINTS.items():
        if not checkpoint.is_file():
            raise FileNotFoundError(checkpoint)
        observed = sha256_file(checkpoint)
        if observed != EXPECTED_CHECKPOINT_HASHES[system]:
            raise RuntimeError(
                f"Checkpoint hash has drifted for {system}: {observed}"
            )
    validate_itercqr_checkpoint(ITERCQR_MODEL_DIR)
    observed_itercqr_hash = sha256_file(
        ITERCQR_MODEL_DIR / "pytorch_model.bin"
    )
    if observed_itercqr_hash != EXPECTED_ITERCQR_MODEL_SHA256:
        raise RuntimeError(
            f"IterCQR weight hash has drifted: {observed_itercqr_hash}"
        )
    observed_serializer_hash = sha256_text(
        inspect.getsource(serialize_itercqr_input)
    )
    if observed_serializer_hash != EXPECTED_SERIALIZER_SHA256:
        raise RuntimeError(
            f"IterCQR serializer hash has drifted: {observed_serializer_hash}"
        )
    pretrained_manifest_path = PRETRAINED_CHECKPOINT.parent / "manifest.json"
    pretrained_manifest = json.loads(
        pretrained_manifest_path.read_text(encoding="utf-8")
    )
    config = pretrained_manifest["config"]
    required = {
        "training_recipe_version": "independent_hf_full_v2",
        "label_space": "collapsed",
        "decoder": "crf",
        "token_loss_weight": 1.0,
        "crf_loss_weight": 1.0,
    }
    for key, value in required.items():
        if config.get(key) != value:
            raise RuntimeError(
                f"NB05 manifest field {key} has drifted: {config.get(key)!r}"
            )
    for arm, checkpoint in (
        ("imitation_control", CONTROL_CHECKPOINT),
        ("bm25_treatment", TREATMENT_CHECKPOINT),
    ):
        manifest = json.loads(
            (checkpoint.parent / "manifest.json").read_text(
                encoding="utf-8"
            )
        )
        if manifest.get("complete") is not True:
            raise RuntimeError(f"NB06 run is incomplete: {arm}")
        arm_config = manifest["config"]
        expected_fields = {
            "arm": arm,
            "epochs": 6,
            "start_checkpoint_sha256": EXPECTED_CHECKPOINT_HASHES[
                "pretrained"
            ],
            "training_recipe_version": POST_TRAINING_RECIPE_VERSION,
            "loss": "weighted_token_CE_plus_token_normalized_CRF_NLL",
            "token_loss_weight": 1.0,
            "crf_loss_weight": 1.0,
        }
        for key, value in expected_fields.items():
            if arm_config.get(key) != value:
                raise RuntimeError(
                    f"NB06 {arm}: {key} has drifted: "
                    f"{arm_config.get(key)!r}"
                )
    architecture = json.loads(
        ARCHITECTURE_DECISION_PATH.read_text(encoding="utf-8")
    )
    if not (
        architecture.get("locked") is True
        and architecture.get("selected_arm") == "collapsed_crf"
        and int(architecture.get("budget")) == PRIMARY_BUDGET
        and architecture.get("selector_checkpoint_sha256")
        == EXPECTED_CHECKPOINT_HASHES["pretrained"]
    ):
        raise RuntimeError("The frozen NB05 architecture has drifted.")
    fusion = json.loads(
        FUSION_DECISION_PATH.read_text(encoding="utf-8")
    )
    if not (
        fusion.get("locked") is True
        and fusion.get("selected_method") == "rrf10"
        and int(fusion.get("budget")) == PRIMARY_BUDGET
        and fusion.get("views") == ["I", "R", "D"]
    ):
        raise RuntimeError("The frozen NB05 fusion has drifted.")
    return config


def load_or_predict_viterbi(
    system: str,
    *,
    selector_tokenizer: Any,
    pretrained_config: Mapping[str, Any],
    inference_rows: list[dict[str, Any]],
    no_history_ids: list[str],
) -> dict[str, list[dict[str, Any]]]:
    system_dir = RESULT_DIR / "predictions" / system
    output_path = system_dir / "viterbi.jsonl"
    manifest_path = system_dir / "manifest.json"
    expected = {
        "schema_version": 1,
        "system": system,
        "checkpoint_sha256": EXPECTED_CHECKPOINT_HASHES[system],
        "population_sha256": population_sha256,
        "decoder": "viterbi",
    }
    if output_path.is_file() and manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if (
            all(manifest.get(key) == value for key, value in expected.items())
            and manifest.get("output_sha256") == sha256_file(output_path)
        ):
            loaded = read_selected_histories_jsonl(output_path, progress=True)
            if set(loaded) == set(sample_ids):
                return loaded
    model = load_crf_inference_model(
        CHECKPOINTS[system],
        model_name=SELECTOR_CONFIG.model_name,
        model_revision=pretrained_config["model_revision"],
        num_labels=3,
        device=DEVICE,
    )
    predicted = predict_crf_history_candidates(
        inference_rows,
        model=model,
        tokenizer=selector_tokenizer,
        device=DEVICE,
        config=SELECTOR_CONFIG,
        batch_size=SELECTOR_BATCH_SIZE,
        num_samples=0,
        sampling_seed=SEED,
        sampling_seed_mode="per_query_sha256",
        progress=True,
        progress_desc=f"QReCC ROCC {system}",
    )["viterbi"]
    for sample_id in no_history_ids:
        predicted[sample_id] = []
    write_selected_histories_jsonl(
        output_path,
        predicted,
        sample_order=sample_ids,
        progress=True,
    )
    write_json(
        manifest_path,
        {
            **expected,
            "output_sha256": sha256_file(output_path),
            "complete": True,
        },
    )
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return predicted


def retrieval_query_id(query_norm: str) -> str:
    return sha256_text(f"qrecc_retrieval_query_v1\0{query_norm}")


def normalize_exact_query(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def make_query_row(
    *,
    family: str,
    system: str,
    sample_id: str,
    budget: int,
    view: str,
    candidate_index: int,
    query: str,
    input_key: str,
    input_length: int,
    raw_input_length: int,
    was_truncated: bool,
    selected_history_words: int,
) -> dict[str, Any]:
    query_norm = normalize_exact_query(query)
    identity = "\0".join(
        [
            family,
            system,
            sample_id,
            str(int(budget)),
            view,
            str(int(candidate_index)),
            query_norm,
        ]
    )
    return {
        "artifact_id": sha256_text(identity),
        "family": family,
        "system": system,
        "sample_id": sample_id,
        "conversation_source": source_by_sample[sample_id],
        "history_depth": int(depth_by_sample[sample_id]),
        "budget": int(budget),
        "view": view,
        "candidate_index": int(candidate_index),
        "query_id": retrieval_query_id(query_norm),
        "query": str(query),
        "query_norm": query_norm,
        "input_key": str(input_key),
        "input_length": int(input_length),
        "raw_input_length": int(raw_input_length),
        "was_truncated": bool(was_truncated),
        "original_history_words": history_word_count(
            sample_by_id[sample_id].history
        ),
        "selected_history_words": int(selected_history_words),
    }


def build_query_bundle() -> pd.DataFrame:
    pretrained_config = validate_generation_artifacts()
    inference_rows = [
        {
            "sample_id": sample.sample_id,
            "current_query": sample.current_query,
            "history": sample.history,
        }
        for sample in samples
        if sample.history
    ]
    no_history_ids = [sample.sample_id for sample in samples if not sample.history]
    selector_tokenizer = AutoTokenizer.from_pretrained(
        SELECTOR_CONFIG.model_name,
        revision=pretrained_config["model_revision"],
        use_fast=True,
    )
    predictions = {
        system: load_or_predict_viterbi(
            system,
            selector_tokenizer=selector_tokenizer,
            pretrained_config=pretrained_config,
            inference_rows=inference_rows,
            no_history_ids=no_history_ids,
        )
        for system in SYSTEM_ORDER
    }
    del selector_tokenizer
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    tokenizer = load_itercqr_tokenizer(ITERCQR_MODEL_DIR)
    rewrite_pipeline = CachedIterCQRBM25Pipeline(
        config=PIPELINE_CONFIG,
        tokenizer=tokenizer,
    )
    full_histories = {
        sample.sample_id: sample.history for sample in samples
    }
    base_histories = {
        "I": full_histories,
        "pretrained_R": predictions["pretrained"],
    }
    base_serialized, _ = materialize_history_arm_inputs(
        samples=samples,
        histories_by_arm=base_histories,
        arm_order=tuple(base_histories),
        tokenizer=tokenizer,
        budgets=BUDGETS,
        pipeline=rewrite_pipeline,
        progress=True,
        progress_desc="QReCC serialize I/pretrained-R",
    )
    base_rewrites = rewrite_pipeline.materialize_rewrites(
        base_serialized,
        progress_prefix="NB07b base",
    ).serialized_with_rewrite
    post_histories = {
        "imitation_control_e6_R": predictions["imitation_control_e6"],
        "bm25_treatment_e6_R": predictions["bm25_treatment_e6"],
    }
    post_serialized, _ = materialize_history_arm_inputs(
        samples=samples,
        histories_by_arm=post_histories,
        arm_order=tuple(post_histories),
        tokenizer=tokenizer,
        budgets=(PRIMARY_BUDGET,),
        pipeline=rewrite_pipeline,
        progress=True,
        progress_desc="QReCC serialize post-training B64",
    )
    post_rewrites = rewrite_pipeline.materialize_rewrites(
        post_serialized,
        progress_prefix="NB07b post",
    ).serialized_with_rewrite
    main_rewrites = pd.concat(
        [base_rewrites, post_rewrites],
        ignore_index=True,
    )
    main_rewrites["history_depth"] = main_rewrites["sample_id"].map(
        depth_by_sample
    )

    query_rows: list[dict[str, Any]] = []
    system_budgets = {
        "pretrained": BUDGETS,
        "imitation_control_e6": (PRIMARY_BUDGET,),
        "bm25_treatment_e6": (PRIMARY_BUDGET,),
    }
    for system, budgets in system_budgets.items():
        for budget in budgets:
            i_rows = base_rewrites.loc[
                base_rewrites["budget"].eq(budget)
                & base_rewrites["arm"].eq("I")
            ].set_index("sample_id")
            r_arm = f"{system}_R"
            source_frame = (
                base_rewrites if system == "pretrained" else post_rewrites
            )
            r_rows = source_frame.loc[
                source_frame["budget"].eq(budget)
                & source_frame["arm"].eq(r_arm)
            ].set_index("sample_id")
            d_rows = materialize_selector_direct_queries(
                pipeline_results=main_rewrites,
                selector_arm=r_arm,
                budget=budget,
                config=EXPERIMENT_CONFIG,
                device=str(DEVICE),
                progress=True,
            ).set_index("sample_id")
            for sample_id in sample_ids:
                i_source = i_rows.loc[sample_id]
                r_source = r_rows.loc[sample_id]
                no_history = depth_by_sample[sample_id] == 0
                i_query = str(i_source["rewrite"])
                r_query = i_query if no_history else str(r_source["rewrite"])
                d_query = (
                    str(sample_by_id[sample_id].current_query)
                    if no_history
                    else str(d_rows.loc[sample_id, "direct_query"])
                )
                selected_words = (
                    0
                    if no_history
                    else history_word_count(predictions[system][sample_id])
                )
                for view, query, source, words in (
                    (
                        "I",
                        i_query,
                        i_source,
                        history_word_count(sample_by_id[sample_id].history),
                    ),
                    ("R", r_query, r_source, selected_words),
                    ("D", d_query, r_source, selected_words),
                ):
                    query_rows.append(
                        make_query_row(
                            family="rocc",
                            system=system,
                            sample_id=sample_id,
                            budget=budget,
                            view=view,
                            candidate_index=0,
                            query=query,
                            input_key=str(source["input_key"]),
                            input_length=(
                                0 if view == "D" else int(source["input_length"])
                            ),
                            raw_input_length=(
                                0
                                if view == "D"
                                else int(source["raw_input_length"])
                            ),
                            was_truncated=(
                                False
                                if view == "D"
                                else bool(source["was_truncated"])
                            ),
                            selected_history_words=words,
                        )
                    )

    iter_rewriter = load_itercqr_rewriter(
        model_dir=ITERCQR_MODEL_DIR,
        device=DEVICE,
    )
    for budget in BUDGETS:
        i_rows = base_rewrites.loc[
            base_rewrites["budget"].eq(budget)
            & base_rewrites["arm"].eq("I")
        ].sort_values("sample_id", kind="mergesort")
        no_history_rows = i_rows.loc[
            i_rows["sample_id"].map(depth_by_sample).eq(0)
        ]
        generation_rows = i_rows.loc[
            i_rows["sample_id"].map(depth_by_sample).ge(1)
        ]
        for source in no_history_rows.itertuples(index=False):
            for candidate_index, view in enumerate(
                ("greedy", "sample_1", "sample_2")
            ):
                query_rows.append(
                    make_query_row(
                        family="itercqr",
                        system="itercqr",
                        sample_id=str(source.sample_id),
                        budget=budget,
                        view=view,
                        candidate_index=candidate_index,
                        query=str(source.rewrite),
                        input_key=str(source.input_key),
                        input_length=int(source.input_length),
                        raw_input_length=int(source.raw_input_length),
                        was_truncated=bool(source.was_truncated),
                        selected_history_words=0,
                    )
                )
        for start in tqdm(
            range(0, len(generation_rows), REWRITE_BATCH_SIZE),
            desc=f"QReCC IterCQR G+2S B{budget}",
            unit="batch",
        ):
            batch_rows = generation_rows.iloc[
                start : start + REWRITE_BATCH_SIZE
            ]
            generated = iter_rewriter.rewrite_greedy_and_samples_batch(
                rewrite_pipeline.make_rewriter_batch(
                    batch_rows["input_key"].astype(str).tolist()
                ),
                sample_ids=batch_rows["sample_id"].astype(str).tolist(),
                num_samples=NUM_ITER_SAMPLES,
                base_sampling_seed=ITER_SAMPLING_SEED,
                sampling_temperature=ITER_SAMPLING_TEMPERATURE,
                sampling_top_k=ITER_SAMPLING_TOP_K,
                sampling_top_p=ITER_SAMPLING_TOP_P,
            )
            for row_index, source in enumerate(
                batch_rows.itertuples(index=False)
            ):
                candidates = {
                    "greedy": str(generated["greedy"][row_index]),
                    "sample_1": str(generated["sample_1"][row_index]),
                    "sample_2": str(generated["sample_2"][row_index]),
                }
                if normalize_rewrite(candidates["greedy"]) != normalize_rewrite(
                    str(source.rewrite)
                ):
                    raise RuntimeError(
                        f"IterCQR greedy output has drifted for {source.sample_id}, B{budget}."
                    )
                for candidate_index, view in enumerate(
                    ("greedy", "sample_1", "sample_2")
                ):
                    query_rows.append(
                        make_query_row(
                            family="itercqr",
                            system="itercqr",
                            sample_id=str(source.sample_id),
                            budget=budget,
                            view=view,
                            candidate_index=candidate_index,
                            query=candidates[view],
                            input_key=str(source.input_key),
                            input_length=int(source.input_length),
                            raw_input_length=int(source.raw_input_length),
                            was_truncated=bool(source.was_truncated),
                            selected_history_words=history_word_count(
                                sample_by_id[str(source.sample_id)].history
                            ),
                        )
                    )
    del iter_rewriter
    rewrite_pipeline.close()
    del rewrite_pipeline
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    bundle = pd.DataFrame(query_rows).loc[:, BUNDLE_COLUMNS]
    bundle = bundle.sort_values(
        [
            "family",
            "system",
            "budget",
            "sample_id",
            "view",
            "candidate_index",
        ],
        kind="mergesort",
    ).reset_index(drop=True)
    if bundle["artifact_id"].duplicated().any():
        raise RuntimeError("Duplicate artifact_id in the QReCC query bundle.")
    query_identity = bundle.groupby("query_id")["query_norm"].nunique()
    if query_identity.gt(1).any():
        raise RuntimeError("query_id collision in the QReCC query bundle.")
    write_bundle(QUERY_BUNDLE_PATH, bundle)
    duplicate_rows = []
    iter_rows = bundle.loc[
        bundle["family"].eq("itercqr")
        & bundle["history_depth"].ge(1)
    ]
    for budget, group in iter_rows.groupby("budget", observed=True):
        unique_per_query = group.groupby("sample_id")["query_norm"].nunique()
        duplicate_rows.append(
            {
                "budget": int(budget),
                "eligible_queries": int(unique_per_query.size),
                "requested_rewrites": int(len(group)),
                "unique_rewrites": int(unique_per_query.sum()),
                "duplicate_rate": float(
                    (len(group) - unique_per_query.sum()) / len(group)
                ),
                "queries_with_duplicate_share": float(
                    unique_per_query.lt(3).mean()
                ),
            }
        )
    write_frame(
        RESULT_DIR / "query_bundle/itercqr_duplicate_summary.csv",
        pd.DataFrame(duplicate_rows),
    )
    manifest = {
        "schema_version": 1,
        **expected_bundle_identity(),
        "rows": int(len(bundle)),
        "unique_retrieval_queries": int(bundle["query_id"].nunique()),
        "query_bundle_sha256": sha256_file(QUERY_BUNDLE_PATH),
        "complete": True,
        "configuration_selection_performed_on_qrecc": False,
        "latency_measured": False,
    }
    write_json(QUERY_BUNDLE_MANIFEST_PATH, manifest)
    return bundle


if QUERY_BUNDLE_PATH.exists() or QUERY_BUNDLE_MANIFEST_PATH.exists():
    if not bundle_is_valid():
        raise RuntimeError(
            "The existing QReCC query bundle is incomplete or has drifted."
        )
    query_artifacts = read_bundle(QUERY_BUNDLE_PATH)
    bundle_reused = True
else:
    query_artifacts = build_query_bundle()
    bundle_reused = False

bundle_manifest = json.loads(
    QUERY_BUNDLE_MANIFEST_PATH.read_text(encoding="utf-8")
)
bundle_sha256 = str(bundle_manifest["query_bundle_sha256"])
missing_bundle_columns = set(BUNDLE_COLUMNS).difference(
    query_artifacts.columns
)
if missing_bundle_columns:
    raise RuntimeError(
        "Query bundle columns are missing: "
        + ", ".join(sorted(missing_bundle_columns))
    )
if len(query_artifacts) != len(sample_ids) * 30:
    raise RuntimeError(
        f"Unexpected query bundle size: {len(query_artifacts)}"
    )
if query_artifacts["artifact_id"].duplicated().any():
    raise RuntimeError("Duplicate artifact_id in the loaded query bundle.")
if set(query_artifacts["sample_id"].astype(str)) != set(sample_ids):
    raise RuntimeError("The query bundle and QReCC population are not aligned.")

display(
    pd.DataFrame(
        [
            {
                "bundle_reused": bundle_reused,
                "query_rows": len(query_artifacts),
                "unique_retrieval_queries": query_artifacts[
                    "query_id"
                ].nunique(),
                "bundle_sha256": bundle_sha256,
            }
        ]
    )
)
duplicate_path = RESULT_DIR / "query_bundle/itercqr_duplicate_summary.csv"
if duplicate_path.exists():
    display(pd.read_csv(duplicate_path).round(4))


,bundle_reused,query_rows,unique_retrieval_queries,bundle_sha256
0,True,246270,77724,7f47b1e9fd02de046ba1b488b731ec47934360c60b837b...


,budget,eligible_queries,requested_rewrites,unique_rewrites,duplicate_rate,queries_with_duplicate_share
0,64,6852,20556,17596,0.1440,0.3600
1,128,6852,20556,17288,0.1590,0.3933
2,256,6852,20556,17294,0.1587,0.3920
3,512,6852,20556,17295,0.1586,0.3913


The query bundle had already been generated and validated in a previous run and was therefore reused.

For the 6,852 queries with history, the greedy IterCQR rewrite and two sampled rewrites are compared within each query. At B64, 14.4% of the requested rewrites are duplicates and 36.0% of queries contain fewer than three distinct candidates. At larger budgets, the duplicate rate remains about 15.9%, affecting approximately 39% of queries.

Duplicate rewrites produce identical retrieval queries and reduce the number of distinct rankings available to RRF. Larger context budgets do not increase the effective diversity of the sampled IterCQR candidates.

### Persistent backend rankings

Retrieving 76,370 unique queries against the complete QReCC collection is the most expensive stage of NB07b. Rankings are therefore stored independently from metric computation, allowing evaluation and bootstrap comparisons to be repeated without running retrieval again.

BM25 processes the complete index in one run. The ANCE index is divided into two disjoint shard sessions because the full dense index exceeds the available GPU memory. Each session retrieves its local top 1,000 passages. Merging both stores by their original ANCE scores reconstructs the global top 1,000 ranking for every query.

In [5]:
unique_queries = (
    query_artifacts[["query_id", "query", "query_norm"]]
    .drop_duplicates("query_id")
    .sort_values("query_id", kind="mergesort")
    .reset_index(drop=True)
)
if unique_queries.groupby("query_id")["query_norm"].nunique().gt(1).any():
    raise RuntimeError("A query_id refers to multiple query texts.")


def ranking_manifest_valid(
    manifest_path: Path,
    store_path: Path,
    *,
    backend: str,
    session_id: str | None,
) -> bool:
    if not (manifest_path.is_file() and store_path.is_file()):
        return False
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    return bool(
        manifest.get("complete") is True
        and manifest.get("backend") == backend
        and manifest.get("session_id") == session_id
        and manifest.get("query_bundle_sha256") == bundle_sha256
        and manifest.get("d_no_history_query_rule")
        == D_NO_HISTORY_QUERY_RULE
        and int(manifest.get("top_k", 0)) == RETRIEVAL_TOP_K
        and int(manifest.get("queries", 0)) == len(unique_queries)
        and manifest.get("query_deduplication_mode") == "exact"
        and manifest.get("latency_measured") is False
        and manifest.get("store_sha256") == sha256_file(store_path)
    )


def assert_resumable_store(
    manifest_path: Path,
    store_path: Path,
) -> None:
    if manifest_path.exists():
        raise RuntimeError(
            "The existing ranking manifest does not match the frozen "
            f"query bundle: {manifest_path}"
        )
    if not store_path.exists():
        return
    expected_ids = set(unique_queries["query_id"].astype(str))
    with QueryRankingStore(store_path, read_only=True) as partial_store:
        unexpected = set(iter(partial_store)).difference(expected_ids)
    if unexpected:
        raise RuntimeError(
            "The incomplete ranking store contains query IDs from "
            "a different bundle."
        )


backend_ready = False
ranking_store_path: Path | None = None
backend_dir = RESULT_DIR / "backends" / BACKEND
final_manifest_path = backend_dir / "final_manifest.json"
result_frame_paths = {
    "route_metrics": backend_dir / "route_metrics_by_query.csv",
    "route_summary": backend_dir / "route_summary.csv",
    "comparisons": backend_dir / "paired_comparisons.csv",
    "compression_summary": backend_dir / "compression_summary.csv",
    "budget_identity_summary": (
        backend_dir / "budget_identity_summary.csv"
    ),
    "saturation_comparisons": (
        backend_dir / "saturation_comparisons.csv"
    ),
    "depth_summary": backend_dir / "depth_strata.csv",
    "source_summary": backend_dir / "source_strata.csv",
}
canonical_ranking_store = (
    backend_dir / "rankings.sqlite3"
    if BACKEND == "bm25"
    else backend_dir / "merged/rankings.sqlite3"
)


def load_complete_analysis_manifest() -> dict[str, Any] | None:
    if not final_manifest_path.exists():
        return None
    manifest = json.loads(
        final_manifest_path.read_text(encoding="utf-8")
    )
    expected_identity = {
        "schema_version": 1,
        "protocol": "nb07b_qrecc_ood_final_v2",
        "d_no_history_query_rule": D_NO_HISTORY_QUERY_RULE,
        "dataset": "QReCC test",
        "backend": BACKEND,
        "population_sha256": population_sha256,
        "query_bundle_sha256": bundle_sha256,
        "budgets": list(BUDGETS),
        "primary_budget": PRIMARY_BUDGET,
        "rrf_k": RRF_K,
        "retrieval_top_k": RETRIEVAL_TOP_K,
        "checkpoint_sha256": EXPECTED_CHECKPOINT_HASHES,
        "itercqr_model_sha256": EXPECTED_ITERCQR_MODEL_SHA256,
        "serializer_sha256": EXPECTED_SERIALIZER_SHA256,
        "configuration_selection_performed_on_qrecc": False,
        "latency_measured": False,
        "complete": True,
    }
    drift = {
        key: (manifest.get(key), expected)
        for key, expected in expected_identity.items()
        if manifest.get(key) != expected
    }
    if drift:
        raise RuntimeError(
            "The existing final manifest has drifted: "
            + repr(drift)
        )
    recorded_hashes = manifest.get("files", {})
    self_relative = str(final_manifest_path.relative_to(RESULT_DIR))
    if self_relative in recorded_hashes:
        raise RuntimeError("The final manifest must not hash itself.")
    required_relatives = {
        str(path.relative_to(RESULT_DIR))
        for path in result_frame_paths.values()
    }
    if not required_relatives.issubset(recorded_hashes):
        raise RuntimeError("Result files are missing from the final manifest.")
    for relative, expected_hash in recorded_hashes.items():
        path = RESULT_DIR / relative
        if not path.is_file():
            raise RuntimeError(
                f"Complete result artifact is missing: {relative}"
            )
        observed_hash = sha256_file(path)
        if observed_hash != expected_hash:
            raise RuntimeError(
                f"Result artifact has drifted: {relative}"
            )
    if not isinstance(manifest.get("ranking_store_sha256"), str):
        raise RuntimeError("Ranking store hash is missing from the final manifest.")
    return manifest


existing_final_manifest = load_complete_analysis_manifest()
REUSE_COMPLETE_ANALYSIS = existing_final_manifest is not None
if not REUSE_COMPLETE_ANALYSIS:
    backend_dir.mkdir(parents=True, exist_ok=True)

if REUSE_COMPLETE_ANALYSIS:
    ranking_store_path = (
        canonical_ranking_store
        if canonical_ranking_store.is_file()
        else None
    )
    backend_ready = True
elif BACKEND == "bm25":
    store_path = backend_dir / "rankings.sqlite3"
    manifest_path = backend_dir / "manifest.json"
    if not ranking_manifest_valid(
        manifest_path,
        store_path,
        backend="bm25",
        session_id=None,
    ):
        assert_resumable_store(manifest_path, store_path)
        if IN_COLAB:
            dependency_status = ensure_colab_retrieval_dependencies()
            display(pd.DataFrame([dependency_status.to_dict()]))
        status = inspect_bm25_lucene_index(
            BM25_INDEX_DIR,
            source_passages=QRECC_TOTAL_PASSAGES,
            log_file=BM25_LOG_FILE,
        )
        display(pd.DataFrame([status.to_dict()]))
        if not status.loadable:
            raise RuntimeError("The local QReCC BM25 index cannot be loaded.")
        retriever = load_bm25_retriever(
            index_dir=BM25_INDEX_DIR,
            k1=0.82,
            b=0.68,
            retrieval_workers=PIPELINE_CONFIG.retrieval_workers,
        )
        stored = run_query_retrieval_to_store(
            queries=unique_queries,
            retriever=retriever,
            store_path=store_path,
            top_k=RETRIEVAL_TOP_K,
            retrieval_batch_size=RETRIEVAL_BATCH_SIZE,
            store_chunk_size=RETRIEVAL_STORE_CHUNK_SIZE,
            measure_latency=False,
            deduplication_mode="exact",
            progress=True,
            progress_desc="QReCC BM25",
        )
        display(pd.DataFrame([stored.to_dict()]))
        if stored.stored_queries != len(unique_queries):
            raise RuntimeError("The BM25 ranking store is incomplete.")
        close = getattr(retriever, "close", None)
        if callable(close):
            close()
        del retriever
        write_json(
            manifest_path,
            {
                "schema_version": 1,
                "backend": "bm25",
                "d_no_history_query_rule": D_NO_HISTORY_QUERY_RULE,
                "session_id": None,
                "query_bundle_sha256": bundle_sha256,
                "queries": len(unique_queries),
                "top_k": RETRIEVAL_TOP_K,
                "k1": 0.82,
                "b": 0.68,
                "query_deduplication_mode": "exact",
                "store_sha256": sha256_file(store_path),
                "complete": True,
                "latency_measured": False,
            },
        )
    ranking_store_path = store_path
    backend_ready = True
elif ranking_manifest_valid(
    backend_dir / "merged/manifest.json",
    backend_dir / "merged/rankings.sqlite3",
    backend="ance",
    session_id=None,
):
    ranking_store_path = backend_dir / "merged/rankings.sqlite3"
    backend_ready = True
else:
    session_dir = backend_dir / QRECC_ANCE_SESSION_ID
    session_dir.mkdir(parents=True, exist_ok=True)
    store_path = session_dir / "rankings.sqlite3"
    manifest_path = session_dir / "manifest.json"
    start_shard, end_shard = qrecc_ance_session_shard_range(
        QRECC_ANCE_SESSION_ID
    )
    if not ranking_manifest_valid(
        manifest_path,
        store_path,
        backend="ance_partial",
        session_id=QRECC_ANCE_SESSION_ID,
    ):
        assert_resumable_store(manifest_path, store_path)
        if IN_COLAB:
            dependency_status = ensure_colab_retrieval_dependencies()
            display(pd.DataFrame([dependency_status.to_dict()]))
        # Session partitioning is unchanged; shards may be local or Drive-linked.
        dense_summary = summarize_dense_faiss_shards(
            DENSE_INDEX_DIR, start_shard=start_shard, end_shard=end_shard,
        )
        capacity = check_dense_gpu_capacity(
            dense_summary,
            device=str(DEVICE),
            index_dtype="float32",
            reserve_gib=4.0,
        )
        display(pd.DataFrame([capacity.__dict__]))
        if not capacity.fits:
            raise RuntimeError("The canonical float32 ANCE session does not fit on the GPU.")
        retriever = load_dense_ance_torch_retriever(
            index_dir=dense_summary.index_dir,
            device=str(DEVICE),
            index_dtype="float32",
            start_shard=start_shard,
            end_shard=end_shard,
            chunk_size=50_000,
            query_batch_size=64,
            progress=True,
            status_log=(
                PROJECT_ROOT / f"experiments/logs/ance_index_load_status_{QRECC_ANCE_SESSION_ID}.log"
            ),
        )
        stored = run_query_retrieval_to_store(
            queries=unique_queries,
            retriever=retriever,
            store_path=store_path,
            top_k=RETRIEVAL_TOP_K,
            retrieval_batch_size=RETRIEVAL_BATCH_SIZE,
            store_chunk_size=RETRIEVAL_STORE_CHUNK_SIZE,
            measure_latency=False,
            deduplication_mode="exact",
            progress=True,
            progress_desc=f"QReCC ANCE {QRECC_ANCE_SESSION_ID}",
        )
        display(pd.DataFrame([stored.to_dict()]))
        if stored.stored_queries != len(unique_queries):
            raise RuntimeError("The ANCE partial store is incomplete.")
        close = getattr(retriever, "close", None)
        if callable(close):
            close()
        del retriever
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        write_json(
            manifest_path,
            {
                "schema_version": 1,
                "backend": "ance_partial",
                "d_no_history_query_rule": D_NO_HISTORY_QUERY_RULE,
                "session_id": QRECC_ANCE_SESSION_ID,
                "start_shard": start_shard,
                "end_shard": end_shard,
                "query_bundle_sha256": bundle_sha256,
                "queries": len(unique_queries),
                "top_k": RETRIEVAL_TOP_K,
                "query_deduplication_mode": "exact",
                "store_sha256": sha256_file(store_path),
                "complete": True,
                "latency_measured": False,
            },
        )

    session_paths = {
        session: backend_dir / session / "rankings.sqlite3"
        for session in ("session_1", "session_2")
    }
    session_manifests = {
        session: backend_dir / session / "manifest.json"
        for session in ("session_1", "session_2")
    }
    sessions_complete = all(
        ranking_manifest_valid(
            session_manifests[session],
            session_paths[session],
            backend="ance_partial",
            session_id=session,
        )
        for session in ("session_1", "session_2")
    )
    if sessions_complete:
        merged_dir = backend_dir / "merged"
        merged_store = merged_dir / "rankings.sqlite3"
        merged_manifest = merged_dir / "manifest.json"
        if not ranking_manifest_valid(
            merged_manifest,
            merged_store,
            backend="ance",
            session_id=None,
        ):
            if merged_manifest.exists() or merged_store.exists():
                raise RuntimeError(
                    "The existing ANCE merge does not match the frozen "
                    "query bundle."
                )
            merged = merge_query_ranking_stores(
                [
                    session_paths["session_1"],
                    session_paths["session_2"],
                ],
                output_path=merged_store,
                top_k=RETRIEVAL_TOP_K,
                progress=True,
            )
            display(pd.DataFrame([merged.to_dict()]))
            write_json(
                merged_manifest,
                {
                    "schema_version": 1,
                    "backend": "ance",
                    "d_no_history_query_rule": D_NO_HISTORY_QUERY_RULE,
                    "session_id": None,
                    "sessions": ["session_1", "session_2"],
                    "query_bundle_sha256": bundle_sha256,
                    "queries": len(unique_queries),
                    "top_k": RETRIEVAL_TOP_K,
                    "query_deduplication_mode": "exact",
                    "store_sha256": sha256_file(merged_store),
                    "complete": True,
                    "latency_measured": False,
                },
            )
        ranking_store_path = merged_store
        backend_ready = True
    else:
        print(
            f"{QRECC_ANCE_SESSION_ID} is complete. "
            "Start the notebook with the other session. The merge and "
            "evaluation will then run automatically."
        )

display(
    pd.DataFrame(
        [
            {
                "backend": BACKEND,
                "backend_ready": backend_ready,
                "analysis_reused": REUSE_COMPLETE_ANALYSIS,
                "ranking_store": (
                    str(ranking_store_path) if ranking_store_path else None
                ),
            }
        ]
    )
)


,backend,backend_ready,analysis_reused,ranking_store
0,ance,True,False,/experiments/...


The selected backend ranking store and completed analysis were already available and were therefore reused without repeating retrieval. The BM25 store contains rankings from the complete sparse index, whereas the ANCE result reuses the merged top-1,000 rankings from both completed shard sessions.

### Route-level retrieval evaluation

ROCC can affect retrieval through three mechanisms. At each budget $B$, $I_B$ rewrites the current query with recency-truncated history. $R_B$ rewrites the current query with ROCC-selected history. $D_B$ detokenizes the same budgeted input used by $R_B$ and submits that text directly to retrieval. Evaluating these views individually reveals where retrieval gains originate. Their fixed RRF10 combinations test whether the views retrieve complementary relevant passages.

IterCQR-1 provides the deterministic rewriting baseline. IterCQR-G+2S-RRF10 tests whether two sampled rewrites add useful retrieval evidence. Identical query texts are collapsed before fusion so that duplicate candidates do not receive additional RRF weight.

Every route is evaluated separately for all 8,209 queries. MRR and nDCG@3 measure early ranking quality, while recall at 10, 100, and 1,000 shows how much relevant evidence remains available deeper in the ranking.

In [6]:
def frame_summary(
    frame: pd.DataFrame,
    groups: Sequence[str],
) -> pd.DataFrame:
    return (
        frame.groupby(list(groups), observed=True)
        .agg(
            n=("sample_id", "nunique"),
            **{metric: (metric, "mean") for metric in METRIC_ORDER},
        )
        .reset_index()
    )


route_metrics = pd.DataFrame()
route_summary = pd.DataFrame()
if REUSE_COMPLETE_ANALYSIS:
    route_metrics = pd.read_csv(result_frame_paths["route_metrics"])
    route_summary = pd.read_csv(result_frame_paths["route_summary"])
    required_metric_columns = {
        "system", "budget", "sample_id", "arm", *METRIC_ORDER,
    }
    required_summary_columns = {
        "system", "budget", "arm", "n", *METRIC_ORDER,
    }
    if not required_metric_columns.issubset(route_metrics.columns):
        raise RuntimeError("Stored route metrics are missing columns.")
    if not required_summary_columns.issubset(route_summary.columns):
        raise RuntimeError("The stored route summary is missing columns.")
    route_metrics["sample_id"] = route_metrics["sample_id"].astype(str)
    route_metrics["budget"] = route_metrics["budget"].astype(int)
    route_summary["budget"] = route_summary["budget"].astype(int)
    route_keys = ["system", "budget", "arm", "sample_id"]
    if (
        len(route_metrics) != len(sample_ids) * 50
        or route_metrics.duplicated(route_keys).any()
        or len(route_summary) != 50
        or not route_summary["n"].eq(len(sample_ids)).all()
    ):
        raise RuntimeError("Stored route results are incomplete.")
    display(
        route_summary.loc[
            route_summary["budget"].eq(PRIMARY_BUDGET)
        ].sort_values(["system", "arm"], kind="mergesort").round(4)
    )
elif backend_ready:
    rocc_view_sets = {
        "I": ("I",),
        "R": ("R",),
        "D": ("D",),
        "I+R": ("I", "R"),
        "I+D": ("I", "D"),
        "R+D": ("R", "D"),
        "I+R+D": ("I", "R", "D"),
    }
    itercqr_view_sets = {
        "IterCQR-1": ("greedy",),
        "IterCQR-G+2S-RRF10": ("greedy", "sample_1", "sample_2"),
    }
    with QueryRankingStore(
        ranking_store_path,
        read_only=True,
    ) as rankings:
        rocc_metrics = evaluate_fixed_rrf_views(
            query_views=query_artifacts.loc[
                query_artifacts["family"].eq("rocc")
            ],
            hits_by_query_id=rankings,
            gold_by_sample=gold_by_sample,
            view_sets=rocc_view_sets,
            group_columns=("system", "budget"),
            rrf_k=RRF_K,
            top_k=RETRIEVAL_TOP_K,
            ks=EVAL_KS,
            measure_latency=False,
            progress=True,
        )
        itercqr_metrics = evaluate_fixed_rrf_views(
            query_views=query_artifacts.loc[
                query_artifacts["family"].eq("itercqr")
            ],
            hits_by_query_id=rankings,
            gold_by_sample=gold_by_sample,
            view_sets=itercqr_view_sets,
            group_columns=("system", "budget"),
            rrf_k=RRF_K,
            top_k=RETRIEVAL_TOP_K,
            ks=EVAL_KS,
            measure_latency=False,
            progress=True,
        )
    route_metrics = pd.concat(
        [rocc_metrics, itercqr_metrics],
        ignore_index=True,
    )
    route_summary = frame_summary(
        route_metrics,
        ("system", "budget", "arm"),
    )
    write_frame(backend_dir / "route_metrics_by_query.csv", route_metrics)
    write_frame(backend_dir / "route_summary.csv", route_summary)
    display(
        route_summary.loc[
            route_summary["budget"].eq(PRIMARY_BUDGET)
        ].sort_values(["system", "arm"], kind="mergesort").round(4)
    )
else:
    print("Routes will be evaluated after both ANCE sessions are complete.")


,system,budget,arm,n,MRR,nDCG@3,R@10,R@100,R@1000
0,bm25_treatment_e6,64,D,8209,0.4261,0.3975,0.6402,0.8174,0.8940
1,bm25_treatment_e6,64,I,8209,0.3775,0.3522,0.5836,0.7629,0.8507
2,bm25_treatment_e6,64,I+D,8209,0.4259,0.3967,0.6608,0.8453,0.9149
3,bm25_treatment_e6,64,I+R,8209,0.3977,0.3690,0.6342,0.8234,0.9026
4,bm25_treatment_e6,64,I+R+D,8209,0.4325,0.4037,0.6706,0.8560,0.9240
5,bm25_treatment_e6,64,R,8209,0.3894,0.3614,0.6085,0.7888,0.8795
6,bm25_treatment_e6,64,R+D,8209,0.4220,0.3934,0.6629,0.8447,0.9172
7,imitation_control_e6,64,D,8209,0.3995,0.3718,0.5998,0.7797,0.8659
8,imitation_control_e6,64,I,8209,0.3775,0.3522,0.5836,0.7629,0.8507
9,imitation_control_e6,64,I+D,8209,0.4138,0.3842,0.6488,0.8357,0.9069


**BM25.** At B64, the base selector gives 0.4391 MRR for `I`, 0.4144 for `R`, 0.4794 for `D`, and 0.4915 for `I+R+D`. IterCQR-G+2S reaches 0.4523. The final fusion therefore exceeds the matched-budget sampled-rewrite baseline by 0.0391 MRR. Treatment gives 0.4283 for `R` and 0.4937 for `I+R+D`, compared with 0.4014 and 0.4854 for the imitation control.

**ANCE.** At B64, the base selector gives 0.3775 MRR for `I`, 0.3799 for `R`, 0.4100 for `D`, and 0.4233 for `I+R+D`. IterCQR-G+2S reaches 0.3842. The final fusion therefore exceeds the matched-budget sampled-rewrite baseline by 0.0391 MRR. Treatment gives 0.3894 for `R` and 0.4325 for `I+R+D`, compared with 0.3754 and 0.4211 for the imitation control.

**BM25 direct retrieval and coverage.** Base `D` has 0.4794 MRR, versus 0.4144 after rewriting. `I+D` reaches 0.4909. Relative to `I`, `I+R+D` gains +0.0524 MRR and +0.0704 $\mathrm{Recall}@1000$. Treatment `D` reaches 0.4794, compared with 0.4725 for control.

**ANCE direct retrieval and coverage.** Base `D` has 0.4100 MRR, versus 0.3799 after rewriting. `I+D` reaches 0.4180. Relative to `I`, `I+R+D` gains +0.0458 MRR and +0.0676 $\mathrm{Recall}@1000$. Treatment `D` reaches 0.4261, compared with 0.3995 for control.

The values use final top-1000 retrieval. Saved final-result tables below cover both backends; original experiment outputs retain the backend of their last execution.

### Comparisons

The comparisons test whether the locked ROCC fusion transfers beyond the IterCQR baselines, whether B64 remains effective against IterCQR at B512, and whether BM25-guided post-training improves over the epoch-matched control and pretrained selector.

Separate comparisons for `R`, `D`, and `I+R+D` locate post-training effects before and after fusion. Deltas are calculated as `left − right`, with uncertainty estimated from 10,000 paired bootstrap samples. All comparisons were fixed before inspecting QReCC results.

In [7]:
comparisons = pd.DataFrame()
if REUSE_COMPLETE_ANALYSIS:
    comparisons = pd.read_csv(result_frame_paths["comparisons"])
    required_comparison_columns = {
        "comparison", "metric", "left_system", "left_budget",
        "left_arm", "right_system", "right_budget", "right_arm",
        "n", "delta", "ci95_low", "ci95_high", "seed",
        "replicates",
    }
    if (
        not required_comparison_columns.issubset(comparisons.columns)
        or len(comparisons) != 65
        or set(comparisons["metric"]) != set(METRIC_ORDER)
        or not comparisons["n"].eq(len(sample_ids)).all()
        or not comparisons["replicates"].eq(BOOTSTRAP_REPLICATES).all()
    ):
        raise RuntimeError("Stored paired comparisons are incomplete.")
    display(
        comparisons.loc[comparisons["metric"].eq("MRR")].round(4)
    )
elif backend_ready:
    def route_slice(
        system: str,
        budget: int,
        arm: str,
    ) -> pd.DataFrame:
        selected = route_metrics.loc[
            route_metrics["system"].eq(system)
            & route_metrics["budget"].eq(int(budget))
            & route_metrics["arm"].eq(arm)
        ].set_index("sample_id")
        if len(selected) != len(sample_ids):
            raise RuntimeError(
                f"Incomplete route: {system}, B{budget}, {arm}"
            )
        return selected

    comparison_specs = (
        (
            "pretrained_R_minus_I_B64",
            ("pretrained", 64, "R"),
            ("pretrained", 64, "I"),
        ),
        (
            "pretrained_IRD_minus_IterCQR1_B64",
            ("pretrained", 64, "I+R+D"),
            ("itercqr", 64, "IterCQR-1"),
        ),
        (
            "pretrained_IRD_minus_IterCQR_G2S_B64",
            ("pretrained", 64, "I+R+D"),
            ("itercqr", 64, "IterCQR-G+2S-RRF10"),
        ),
        (
            "pretrained_IRD_B64_minus_IterCQR_G2S_B512",
            ("pretrained", 64, "I+R+D"),
            ("itercqr", 512, "IterCQR-G+2S-RRF10"),
        ),
        (
            "treatment_R_minus_control_R_B64",
            ("bm25_treatment_e6", 64, "R"),
            ("imitation_control_e6", 64, "R"),
        ),
        (
            "treatment_R_minus_pretrained_R_B64",
            ("bm25_treatment_e6", 64, "R"),
            ("pretrained", 64, "R"),
        ),
        (
            "control_R_minus_pretrained_R_B64",
            ("imitation_control_e6", 64, "R"),
            ("pretrained", 64, "R"),
        ),
        (
            "treatment_D_minus_control_D_B64",
            ("bm25_treatment_e6", 64, "D"),
            ("imitation_control_e6", 64, "D"),
        ),
        (
            "treatment_D_minus_pretrained_D_B64",
            ("bm25_treatment_e6", 64, "D"),
            ("pretrained", 64, "D"),
        ),
        (
            "control_D_minus_pretrained_D_B64",
            ("imitation_control_e6", 64, "D"),
            ("pretrained", 64, "D"),
        ),
        (
            "treatment_IRD_minus_control_IRD_B64",
            ("bm25_treatment_e6", 64, "I+R+D"),
            ("imitation_control_e6", 64, "I+R+D"),
        ),
        (
            "treatment_IRD_minus_pretrained_IRD_B64",
            ("bm25_treatment_e6", 64, "I+R+D"),
            ("pretrained", 64, "I+R+D"),
        ),
        (
            "control_IRD_minus_pretrained_IRD_B64",
            ("imitation_control_e6", 64, "I+R+D"),
            ("pretrained", 64, "I+R+D"),
        ),
    )
    rows = []
    for comparison_index, (name, left_spec, right_spec) in enumerate(
        comparison_specs
    ):
        left = route_slice(*left_spec)
        right = route_slice(*right_spec)
        paired_ids = sorted(set(left.index).intersection(right.index))
        if len(paired_ids) != len(sample_ids):
            raise RuntimeError(f"Incomplete pairing: {name}")
        for metric_index, metric in enumerate(METRIC_ORDER):
            deltas = (
                left.loc[paired_ids, metric].to_numpy(dtype=float)
                - right.loc[paired_ids, metric].to_numpy(dtype=float)
            )
            seed = 1_000 + comparison_index * len(METRIC_ORDER) + metric_index
            low, high = paired_bootstrap_mean_ci(
                deltas,
                seed=seed,
                replicates=BOOTSTRAP_REPLICATES,
                progress=True,
                progress_desc=f"QReCC {name} {metric}",
            )
            rows.append(
                {
                    "comparison": name,
                    "metric": metric,
                    "left_system": left_spec[0],
                    "left_budget": left_spec[1],
                    "left_arm": left_spec[2],
                    "right_system": right_spec[0],
                    "right_budget": right_spec[1],
                    "right_arm": right_spec[2],
                    "n": len(deltas),
                    "delta": float(deltas.mean()),
                    "ci95_low": float(low),
                    "ci95_high": float(high),
                    "seed": seed,
                    "replicates": BOOTSTRAP_REPLICATES,
                }
            )
    comparisons = pd.DataFrame(rows)
    write_frame(backend_dir / "paired_comparisons.csv", comparisons)
    display(
        comparisons.loc[comparisons["metric"].eq("MRR")].round(4)
    )


,comparison,metric,left_system,left_budget,left_arm,right_system,right_budget,right_arm,n,delta,ci95_low,ci95_high,seed,replicates
0,pretrained_R_minus_I_B64,MRR,pretrained,64,R,pretrained,64,I,8209,0.0024,-0.0043,0.0091,1000,10000
5,pretrained_IRD_minus_IterCQR1_B64,MRR,pretrained,64,I+R+D,itercqr,64,IterCQR-1,8209,0.0458,0.0403,0.0514,1005,10000
10,pretrained_IRD_minus_IterCQR_G2S_B64,MRR,pretrained,64,I+R+D,itercqr,64,IterCQR-G+2S-RRF10,8209,0.0391,0.0335,0.0446,1010,10000
15,pretrained_IRD_B64_minus_IterCQR_G2S_B512,MRR,pretrained,64,I+R+D,itercqr,512,IterCQR-G+2S-RRF10,8209,0.0177,0.0118,0.0237,1015,10000
20,treatment_R_minus_control_R_B64,MRR,bm25_treatment_e6,64,R,imitation_control_e6,64,R,8209,0.0140,0.0080,0.0200,1020,10000
25,treatment_R_minus_pretrained_R_B64,MRR,bm25_treatment_e6,64,R,pretrained,64,R,8209,0.0095,0.0037,0.0153,1025,10000
30,control_R_minus_pretrained_R_B64,MRR,imitation_control_e6,64,R,pretrained,64,R,8209,-0.0045,-0.0092,0.0004,1030,10000
35,treatment_D_minus_control_D_B64,MRR,bm25_treatment_e6,64,D,imitation_control_e6,64,D,8209,0.0266,0.0211,0.0320,1035,10000
40,treatment_D_minus_pretrained_D_B64,MRR,bm25_treatment_e6,64,D,pretrained,64,D,8209,0.0161,0.0111,0.0211,1040,10000
45,control_D_minus_pretrained_D_B64,MRR,imitation_control_e6,64,D,pretrained,64,D,8209,-0.0105,-0.0148,-0.0065,1045,10000


**BM25.** Base B64 `I+R+D` versus the single B64 IterCQR rewrite gives +0.0524 MRR, 95% CI [+0.0463, +0.0585]. The comparison with B64 IterCQR-G+2S gives +0.0391 MRR, 95% CI [+0.0333, +0.0450]. Against B512 IterCQR-G+2S, the B64 fusion gives +0.0276 MRR, 95% CI [+0.0212, +0.0339]. These are paired comparisons on the same queries; a positive interval excludes zero in favor of the left-hand system.

**ANCE.** Base B64 `I+R+D` versus the single B64 IterCQR rewrite gives +0.0458 MRR, 95% CI [+0.0403, +0.0514]. The comparison with B64 IterCQR-G+2S gives +0.0391 MRR, 95% CI [+0.0335, +0.0446]. Against B512 IterCQR-G+2S, the B64 fusion gives +0.0177 MRR, 95% CI [+0.0118, +0.0237]. These are paired comparisons on the same queries; a positive interval excludes zero in favor of the left-hand system.

**BM25.** Base, imitation-control and corrected-label `I+R+D` reach 0.4915, 0.4854 and 0.4937 MRR. Control minus base gives -0.0061 MRR, 95% CI [-0.0097, -0.0024]; treatment minus base gives +0.0022 MRR, 95% CI [-0.0020, +0.0064]; treatment minus control gives +0.0083 MRR, 95% CI [+0.0040, +0.0126]. For standalone `R`, the corresponding base/control/treatment means are 0.4144/0.4014/0.4283.

**ANCE.** Base, imitation-control and corrected-label `I+R+D` reach 0.4233, 0.4211 and 0.4325 MRR. Control minus base gives -0.0022 MRR, 95% CI [-0.0057, +0.0014]; treatment minus base gives +0.0092 MRR, 95% CI [+0.0049, +0.0135]; treatment minus control gives +0.0114 MRR, 95% CI [+0.0069, +0.0157]. For standalone `R`, the corresponding base/control/treatment means are 0.3799/0.3754/0.3894.

### Compression, budget stability, and transfer

ROCC should remove distracting history while preserving the evidence needed for retrieval. The compression summary quantifies how much context the frozen pretrained selector retains. Queries without history are excluded when assessing empty selections.

A larger budget is useful only when it changes the serialized input or retrieval query and produces a corresponding quality gain. Cross-budget identity rates capture the first condition, while paired MRR differences test the second for `R` and `D`. This connects additional context capacity to retrieval value instead of treating a larger budget as inherently beneficial.

History-depth strata show where context selection becomes important as conversations grow. Source strata examine whether the same behavior transfers across NQ, QuAC, and TREC. These subgroup results are descriptive because no paired confidence intervals are computed within the strata. No QReCC result changes the frozen selector, fusion rule, or primary budget.

In [8]:
compression_summary = pd.DataFrame()
budget_identity_summary = pd.DataFrame()
saturation_comparisons = pd.DataFrame()
depth_summary = pd.DataFrame()
source_summary = pd.DataFrame()
if REUSE_COMPLETE_ANALYSIS:
    compression_summary = pd.read_csv(
        result_frame_paths["compression_summary"]
    )
    budget_identity_summary = pd.read_csv(
        result_frame_paths["budget_identity_summary"]
    )
    saturation_comparisons = pd.read_csv(
        result_frame_paths["saturation_comparisons"]
    )
    depth_summary = pd.read_csv(result_frame_paths["depth_summary"])
    source_summary = pd.read_csv(result_frame_paths["source_summary"])
    if (
        len(compression_summary) != len(BUDGETS)
        or set(compression_summary["budget"].astype(int)) != set(BUDGETS)
        or not compression_summary["n"].eq(len(sample_ids)).all()
        or len(budget_identity_summary) != 6
        or not budget_identity_summary["n"].eq(len(sample_ids)).all()
        or len(saturation_comparisons) != 30
        or not saturation_comparisons["n"].eq(len(sample_ids)).all()
        or not saturation_comparisons["replicates"].eq(
            BOOTSTRAP_REPLICATES
        ).all()
        or len(depth_summary) != 35
        or len(source_summary) != 15
    ):
        raise RuntimeError("Stored descriptive tables are incomplete.")
    display(compression_summary.round(4))
    display(budget_identity_summary.round(4))
    display(
        saturation_comparisons.loc[
            saturation_comparisons["metric"].eq("MRR")
        ].round(4)
    )
    display(depth_summary.round(4))
    display(source_summary.round(4))
elif backend_ready:
    pretrained_r = query_artifacts.loc[
        query_artifacts["family"].eq("rocc")
        & query_artifacts["system"].eq("pretrained")
        & query_artifacts["view"].eq("R")
    ].copy()
    compression_summary = (
        pretrained_r.groupby("budget", observed=True)
        .agg(
            n=("sample_id", "nunique"),
            original_history_words=("original_history_words", "sum"),
            selected_history_words=("selected_history_words", "sum"),
            mean_selected_history_words=("selected_history_words", "mean"),
            mean_input_length=("input_length", "mean"),
            truncation_rate=("was_truncated", "mean"),
            empty_selection_rate=(
                "selected_history_words",
                lambda values: float(np.mean(np.asarray(values) == 0)),
            ),
        )
        .reset_index()
    )
    compression_summary["keep_rate"] = (
        compression_summary["selected_history_words"]
        / compression_summary["original_history_words"]
    )
    compression_summary["compression_rate"] = 1.0 - compression_summary[
        "keep_rate"
    ]
    write_frame(backend_dir / "compression_summary.csv", compression_summary)

    identity_rows = []
    for view in ("R", "D"):
        view_rows = query_artifacts.loc[
            query_artifacts["family"].eq("rocc")
            & query_artifacts["system"].eq("pretrained")
            & query_artifacts["view"].eq(view)
        ]
        for left_budget, right_budget in ((128, 64), (256, 128), (512, 256)):
            left = view_rows.loc[
                view_rows["budget"].eq(left_budget)
            ].set_index("sample_id")
            right = view_rows.loc[
                view_rows["budget"].eq(right_budget)
            ].set_index("sample_id")
            paired_ids = sorted(set(left.index).intersection(right.index))
            if len(paired_ids) != len(sample_ids):
                raise RuntimeError("Incomplete cross-budget input comparison.")
            identity_rows.append(
                {
                    "view": view,
                    "comparison": f"B{left_budget} vs B{right_budget}",
                    "n": len(paired_ids),
                    "identical_input_share": float(
                        np.mean(
                            left.loc[paired_ids, "input_key"].to_numpy()
                            == right.loc[paired_ids, "input_key"].to_numpy()
                        )
                    ),
                    "identical_query_share": float(
                        np.mean(
                            left.loc[paired_ids, "query_norm"].to_numpy()
                            == right.loc[paired_ids, "query_norm"].to_numpy()
                        )
                    ),
                }
            )
    budget_identity_summary = pd.DataFrame(identity_rows)
    write_frame(
        backend_dir / "budget_identity_summary.csv",
        budget_identity_summary,
    )

    saturation_rows = []
    consecutive = ((128, 64), (256, 128), (512, 256))
    for arm_index, arm in enumerate(("R", "D")):
        for pair_index, (left_budget, right_budget) in enumerate(consecutive):
            left = route_slice("pretrained", left_budget, arm)
            right = route_slice("pretrained", right_budget, arm)
            paired_ids = sorted(left.index)
            for metric_index, metric in enumerate(METRIC_ORDER):
                deltas = (
                    left.loc[paired_ids, metric].to_numpy(dtype=float)
                    - right.loc[paired_ids, metric].to_numpy(dtype=float)
                )
                seed = (
                    2_000
                    + (arm_index * len(consecutive) + pair_index)
                    * len(METRIC_ORDER)
                    + metric_index
                )
                low, high = paired_bootstrap_mean_ci(
                    deltas,
                    seed=seed,
                    replicates=BOOTSTRAP_REPLICATES,
                    progress=True,
                    progress_desc=(
                        f"QReCC saturation {arm} B{left_budget}-B{right_budget} {metric}"
                    ),
                )
                saturation_rows.append(
                    {
                        "arm": arm,
                        "comparison": f"B{left_budget} - B{right_budget}",
                        "metric": metric,
                        "n": len(deltas),
                        "delta": float(deltas.mean()),
                        "ci95_low": float(low),
                        "ci95_high": float(high),
                        "seed": seed,
                        "replicates": BOOTSTRAP_REPLICATES,
                    }
                )
    saturation_comparisons = pd.DataFrame(saturation_rows)
    write_frame(
        backend_dir / "saturation_comparisons.csv",
        saturation_comparisons,
    )

    def depth_bin(depth: int) -> str:
        if depth == 0:
            return "d00"
        if depth == 1:
            return "d01"
        if depth <= 4:
            return "d02_04"
        if depth <= 6:
            return "d05_06"
        if depth <= 8:
            return "d07_08"
        if depth <= 10:
            return "d09_10"
        if depth <= 14:
            return "d11_14"
        return "d15_plus"

    descriptive = route_metrics.loc[
        route_metrics["budget"].eq(PRIMARY_BUDGET)
        & (
            (
                route_metrics["system"].eq("pretrained")
                & route_metrics["arm"].isin(["I", "R", "D", "I+R+D"])
            )
            | (
                route_metrics["system"].eq("itercqr")
                & route_metrics["arm"].eq("IterCQR-G+2S-RRF10")
            )
        )
    ].copy()
    descriptive["history_depth"] = descriptive["sample_id"].map(
        depth_by_sample
    )
    descriptive["depth_bin"] = descriptive["history_depth"].map(depth_bin)
    descriptive["conversation_source"] = descriptive["sample_id"].map(
        source_by_sample
    )
    depth_summary = frame_summary(
        descriptive,
        ("system", "arm", "depth_bin"),
    )
    source_summary = frame_summary(
        descriptive,
        ("system", "arm", "conversation_source"),
    )
    write_frame(backend_dir / "depth_strata.csv", depth_summary)
    write_frame(backend_dir / "source_strata.csv", source_summary)
    display(compression_summary.round(4))
    display(budget_identity_summary.round(4))
    display(
        saturation_comparisons.loc[
            saturation_comparisons["metric"].eq("MRR")
        ].round(4)
    )
    display(depth_summary.round(4))
    display(source_summary.round(4))


,budget,n,original_history_words,selected_history_words,mean_selected_history_words,mean_input_length,truncation_rate,empty_selection_rate,keep_rate,compression_rate
0,64,8209,599621,150349,18.3151,40.9440,0.1941,0.1653,0.2507,0.7493
1,128,8209,599621,150349,18.3151,42.6375,0.1191,0.1653,0.2507,0.7493
2,256,8209,599621,150349,18.3151,42.6398,0.1189,0.1653,0.2507,0.7493
3,512,8209,599621,150349,18.3151,42.6398,0.1189,0.1653,0.2507,0.7493


,view,comparison,n,identical_input_share,identical_query_share
0,R,B128 vs B64,8209,0.8762,0.9018
1,R,B256 vs B128,8209,0.9996,0.9998
2,R,B512 vs B256,8209,1.0000,1.0000
3,D,B128 vs B64,8209,0.8762,0.8777
4,D,B256 vs B128,8209,0.9996,0.9996
5,D,B512 vs B256,8209,1.0000,1.0000


,arm,comparison,metric,n,delta,ci95_low,ci95_high,seed,replicates
0,R,B128 - B64,MRR,8209,0.0011,-0.0009,0.0031,2000,10000
5,R,B256 - B128,MRR,8209,-0.0001,-0.0002,0.0000,2005,10000
10,R,B512 - B256,MRR,8209,0.0000,0.0000,0.0000,2010,10000
15,D,B128 - B64,MRR,8209,0.0028,0.0017,0.0040,2015,10000
20,D,B256 - B128,MRR,8209,0.0000,0.0000,0.0000,2020,10000
25,D,B512 - B256,MRR,8209,0.0000,0.0000,0.0000,2025,10000


,system,arm,depth_bin,n,MRR,nDCG@3,R@10,R@100,R@1000
0,itercqr,IterCQR-G+2S-RRF10,d00,1357,0.3839,0.3503,0.6205,0.8135,0.9022
1,itercqr,IterCQR-G+2S-RRF10,d01,1343,0.4496,0.4214,0.6594,0.8314,0.8998
2,itercqr,IterCQR-G+2S-RRF10,d02_04,3625,0.3892,0.3603,0.5970,0.7750,0.8578
3,itercqr,IterCQR-G+2S-RRF10,d05_06,1154,0.3375,0.3150,0.5462,0.7406,0.8351
4,itercqr,IterCQR-G+2S-RRF10,d07_08,547,0.3266,0.3053,0.5440,0.7742,0.8445
5,itercqr,IterCQR-G+2S-RRF10,d09_10,167,0.2949,0.2793,0.5180,0.7455,0.8084
6,itercqr,IterCQR-G+2S-RRF10,d11_14,16,0.0938,0.0625,0.1875,0.6250,0.8125
7,pretrained,D,d00,1357,0.4215,0.3862,0.6524,0.8466,0.9166
8,pretrained,D,d01,1343,0.4591,0.4324,0.6525,0.8122,0.8819
9,pretrained,D,d02_04,3625,0.4132,0.3859,0.6180,0.7818,0.8693


,system,arm,conversation_source,n,MRR,nDCG@3,R@10,R@100,R@1000
0,itercqr,IterCQR-G+2S-RRF10,nq,1442,0.4355,0.4056,0.6170,0.7826,0.8665
1,itercqr,IterCQR-G+2S-RRF10,quac,6396,0.3798,0.3527,0.6005,0.7896,0.8679
2,itercqr,IterCQR-G+2S-RRF10,trec,371,0.2609,0.2289,0.4811,0.7116,0.8495
3,pretrained,D,nq,1442,0.4138,0.3832,0.5912,0.7704,0.8601
4,pretrained,D,quac,6396,0.4170,0.3902,0.6308,0.7977,0.8785
5,pretrained,D,trec,371,0.2745,0.2477,0.4879,0.7084,0.8495
6,pretrained,I,nq,1442,0.4252,0.3994,0.6003,0.7682,0.8530
7,pretrained,I,quac,6396,0.3734,0.3486,0.5865,0.7654,0.8512
8,pretrained,I,trec,371,0.2631,0.2305,0.4690,0.6981,0.8333
9,pretrained,I+R+D,nq,1442,0.4509,0.4221,0.6448,0.8225,0.9002


ROCC retains 150,349 of 599,621 history words, reducing the conversational context by 74.9% to an average of 18.3 selected words per query. The reported 16.5% empty-selection rate corresponds exactly to the 1,357 queries without history. None of the 6,852 history-bearing queries receives an empty selection. These compression and serialization statistics are shared by BM25 and ANCE because both backends evaluate the identical frozen query bundle.

Increasing the budget from B64 to B128 changes only 9.8% of `R` queries and 12.2% of `D` queries. From B128 onward, inputs and queries are effectively identical, reaching exact identity between B256 and B512.

**BM25 budget behavior.** The B64-to-B128 changes produce only small MRR differences whose confidence intervals include zero. Additional capacity therefore rarely changes the retrieval evidence after B64 and provides no statistically supported BM25 improvement.

**ANCE budget behavior (retriever OOD).** From B64 to B128, `R` changes by +0.0011 MRR with a 95% CI of [−0.0009, +0.0031]. `D` improves by +0.0028 with [+0.0017, +0.0040]. From B128 onward, queries and MRR are effectively unchanged, with exact identity between B256 and B512.

**Cross-backend budget interpretation.** Both backends show ROCC saturation after B128. ANCE reveals a small, statistically supported B64-to-B128 gain for `D`; subsequent budget increases do not change its MRR.

**BM25 depth strata.** At zero history, `I` and `R` both reach 0.2650 MRR, but `D` reaches 0.3410 and `I+R+D` 0.3302. Only the I/R equality follows from their identical no-history rewrite input. Base fusion at `d01` scores 0.5402. Base fusion at `d02_04` scores 0.5319. Base fusion at `d05_06` scores 0.5016. Base fusion at `d07_08` scores 0.4979. Base fusion at `d09_10` scores 0.4558. The complete saved table also includes the small terminal stratum; its estimate is descriptive.

**ANCE depth strata.** At zero history, `I` and `R` both reach 0.3839 MRR, but `D` reaches 0.4215 and `I+R+D` 0.4162. Only the I/R equality follows from their identical no-history rewrite input. Base fusion at `d01` scores 0.4688. Base fusion at `d02_04` scores 0.4334. Base fusion at `d05_06` scores 0.3946. Base fusion at `d07_08` scores 0.3612. Base fusion at `d09_10` scores 0.3243. The complete saved table also includes the small terminal stratum; its estimate is descriptive.

**BM25 source strata.** Base I+R+D minus IterCQR-G+2S gives nq: +0.0426 MRR; quac: +0.0399 MRR; trec: +0.0125 MRR. These subgroup differences are descriptive; no paired source-stratum intervals are reported.

**ANCE source strata.** Base I+R+D minus IterCQR-G+2S gives nq: +0.0153 MRR; quac: +0.0441 MRR; trec: +0.0433 MRR. These subgroup differences are descriptive; no paired source-stratum intervals are reported.

### Current-query-only IterCQR baseline

This baseline removes the complete conversation history before B64 IterCQR rewriting. It measures the out-of-domain retrieval signal available from the current query alone. The persisted subartifact is reused only when its population, model, serializer, query bundle, ranking store, and metric hashes remain valid. Missing QReCC ANCE rankings follow the same two-session shard protocol as the main evaluation.

In [9]:
current_query_only_result = load_or_compute_current_query_only(
    result_dir=RESULT_DIR,
    dataset="qrecc",
    samples=samples,
    sample_ids=sample_ids,
    gold_by_sample=gold_by_sample,
    population_sha256=population_sha256,
    model_dir=ITERCQR_MODEL_DIR,
    expected_model_sha256=EXPECTED_ITERCQR_MODEL_SHA256,
    expected_serializer_sha256=EXPECTED_SERIALIZER_SHA256,
    pipeline_config=PIPELINE_CONFIG,
    backend=BACKEND,
    bm25_index_dir=BM25_INDEX_DIR,
    bm25_k1=0.82,
    bm25_b=0.68,
    dense_index_dir=DENSE_INDEX_DIR,
    azure_sas_file=None,
    qrecc_ance_session_id=QRECC_ANCE_SESSION_ID,
    device=str(DEVICE),
    budget=PRIMARY_BUDGET,
    top_k=RETRIEVAL_TOP_K,
    eval_ks=EVAL_KS,
    retrieval_batch_size=RETRIEVAL_BATCH_SIZE,
    store_chunk_size=RETRIEVAL_STORE_CHUNK_SIZE,
    retrieval_workers=PIPELINE_CONFIG.retrieval_workers,
    progress=True,
)
current_query_only_extension = (
    current_query_only_result.extension
)
if current_query_only_result.ready:
    current_query_only_summary = (
        current_query_only_result.summary
    )
    display(current_query_only_summary.round(4))
else:
    current_query_only_summary = pd.DataFrame()
    display(
        Markdown(
            f"**{QRECC_ANCE_SESSION_ID} current-query-only "
            "rankings are complete. Run the other ANCE session "
            "to merge and evaluate them."
        )
    )

if REUSE_COMPLETE_ANALYSIS and current_query_only_extension is not None:
    updated_manifest = dict(existing_final_manifest)
    updated_extensions = dict(
        updated_manifest.get("extensions", {})
    )
    recorded_extension = updated_extensions.get(
        "current_query_only"
    )
    if (
        recorded_extension is not None
        and recorded_extension != current_query_only_extension
    ):
        raise RuntimeError(
            "Current-query-only extension differs in the "
            "main manifest."
        )
    if recorded_extension is None:
        updated_extensions["current_query_only"] = (
            current_query_only_extension
        )
        updated_manifest["extensions"] = updated_extensions
        write_json(final_manifest_path, updated_manifest)
        existing_final_manifest = updated_manifest


,system,arm,budget,n,MRR,nDCG@3,R@10,R@100,R@1000,mean_input_length,max_input_length
0,itercqr,CurrentQueryOnly,64,8209,0.0968,0.0875,0.153,0.2178,0.2668,11.6531,32


### Final manifest

The manifest binds the fixed evaluation configuration and result artifacts through recorded identities and SHA-256 hashes. `configuration_selection_performed_on_qrecc=false` confirms that QReCC did not influence configuration selection.

In [10]:
def display_final_manifest(manifest: dict[str, Any]) -> None:
    manifest_display = dict(manifest)
    manifest_display["recorded_ranking_store"] = (
        manifest_display.pop("ranking_store", None)
    )
    manifest_display["resolved_local_ranking_store"] = (
        str(ranking_store_path)
        if ranking_store_path is not None
        else None
    )
    display(pd.DataFrame([manifest_display]))


if REUSE_COMPLETE_ANALYSIS:
    final_manifest = dict(existing_final_manifest)
    display_final_manifest(final_manifest)
    display(
        Markdown(
            f"**NB07b restored ({BACKEND.upper()}).** "
            "Query generation, retrieval, fusion, and bootstrap "
            "were not rerun."
        )
    )
elif backend_ready:
    result_files = sorted(
        path
        for path in backend_dir.rglob("*")
        if (
            path.is_file()
            and path.suffix != ".sqlite3"
            and path != final_manifest_path
        )
    )
    ranking_manifest_path = (
        backend_dir / "manifest.json"
        if BACKEND == "bm25"
        else backend_dir / "merged/manifest.json"
    )
    ranking_manifest = json.loads(
        ranking_manifest_path.read_text(encoding="utf-8")
    )
    final_manifest = {
        "schema_version": 1,
        "protocol": "nb07b_qrecc_ood_final_v2",
        "d_no_history_query_rule": D_NO_HISTORY_QUERY_RULE,
        "dataset": "QReCC test",
        "backend": BACKEND,
        "population_sha256": population_sha256,
        "query_bundle_sha256": bundle_sha256,
        "ranking_store": str(ranking_store_path.relative_to(RESULT_DIR)),
        "ranking_store_sha256": ranking_manifest["store_sha256"],
        "budgets": list(BUDGETS),
        "primary_budget": PRIMARY_BUDGET,
        "rrf_k": RRF_K,
        "retrieval_top_k": RETRIEVAL_TOP_K,
        "checkpoint_sha256": EXPECTED_CHECKPOINT_HASHES,
        "itercqr_model_sha256": EXPECTED_ITERCQR_MODEL_SHA256,
        "serializer_sha256": EXPECTED_SERIALIZER_SHA256,
        "configuration_selection_performed_on_qrecc": False,
        "latency_measured": False,
        "extensions": {
            **(
                {
                    "current_query_only": (
                        current_query_only_extension
                    )
                }
                if current_query_only_extension is not None
                else {}
            ),
        },
        "files": {
            str(path.relative_to(RESULT_DIR)): sha256_file(path)
            for path in result_files
        },
        "complete": True,
    }
    write_json(final_manifest_path, final_manifest)
    display_final_manifest(final_manifest)
    if BACKEND == "ance":
        display(
            Markdown(
                "**ANCE complete.** Copy `query_bundle/` and "
                "`backends/ance/` from the Drive artifact directory to "
                "`experiments/results/07b_qrecc_ood/`. The local notebook "
                "run then uses the bundle directly for BM25."
            )
        )
else:
    display(
        Markdown(
            f"**{QRECC_ANCE_SESSION_ID} complete.** "
            "Start the notebook in a new Colab session with the other "
            "`ROCC_QRECC_ANCE_SESSION`."
        )
    )


,schema_version,protocol,d_no_history_query_rule,dataset,backend,population_sha256,query_bundle_sha256,ranking_store_sha256,budgets,primary_budget,...,checkpoint_sha256,itercqr_model_sha256,serializer_sha256,configuration_selection_performed_on_qrecc,latency_measured,extensions,files,complete,recorded_ranking_store,resolved_local_ranking_store
0,1,nb07b_qrecc_ood_final_v2,dataset_current_query_without_t5,QReCC test,ance,c9a19e8d6f7d6cf3ab1db89baf5a2d76da7f63e5e87d0a...,7f47b1e9fd02de046ba1b488b731ec47934360c60b837b...,08d1b2ba8bffea2c717674799f4af051095a58ae7f0798...,"[64, 128, 256, 512]",64,...,{'pretrained': 'a07271abefbc37ef08dba4eb94adb0...,1a25ceaf597bf92ff2897cccb22e9c46b807efbf78a323...,49cf2cd0dd6886f6513af55d007076fba1596374c5c180...,False,False,{'current_query_only': {'protocol': 'itercqr_c...,{'backends/ance/budget_identity_summary.csv': ...,True,backends/ance/merged/rankings.sqlite3,/experiments/...


**ANCE complete.** Copy `query_bundle/` and `backends/ance/` from the Drive artifact directory to `experiments/results/07b_qrecc_ood/`. The local notebook run then uses the bundle directly for BM25.

## Conclusion from the frozen final results

**BM25.** At B64, the base selector gives 0.4391 MRR for `I`, 0.4144 for `R`, 0.4794 for `D`, and 0.4915 for `I+R+D`. IterCQR-G+2S reaches 0.4523. The final fusion therefore exceeds the matched-budget sampled-rewrite baseline by 0.0391 MRR. Treatment gives 0.4283 for `R` and 0.4937 for `I+R+D`, compared with 0.4014 and 0.4854 for the imitation control.

**ANCE.** At B64, the base selector gives 0.3775 MRR for `I`, 0.3799 for `R`, 0.4100 for `D`, and 0.4233 for `I+R+D`. IterCQR-G+2S reaches 0.3842. The final fusion therefore exceeds the matched-budget sampled-rewrite baseline by 0.0391 MRR. Treatment gives 0.3894 for `R` and 0.4325 for `I+R+D`, compared with 0.3754 and 0.4211 for the imitation control.

**BM25 direct retrieval and coverage.** Base `D` has 0.4794 MRR, versus 0.4144 after rewriting. `I+D` reaches 0.4909. Relative to `I`, `I+R+D` gains +0.0524 MRR and +0.0704 $\mathrm{Recall}@1000$. Treatment `D` reaches 0.4794, compared with 0.4725 for control.

**ANCE direct retrieval and coverage.** Base `D` has 0.4100 MRR, versus 0.3799 after rewriting. `I+D` reaches 0.4180. Relative to `I`, `I+R+D` gains +0.0458 MRR and +0.0676 $\mathrm{Recall}@1000$. Treatment `D` reaches 0.4261, compared with 0.3995 for control.

The values use final top-1000 retrieval. Saved final-result tables below cover both backends; original experiment outputs retain the backend of their last execution.

**BM25.** Base B64 `I+R+D` versus the single B64 IterCQR rewrite gives +0.0524 MRR, 95% CI [+0.0463, +0.0585]. The comparison with B64 IterCQR-G+2S gives +0.0391 MRR, 95% CI [+0.0333, +0.0450]. Against B512 IterCQR-G+2S, the B64 fusion gives +0.0276 MRR, 95% CI [+0.0212, +0.0339]. These are paired comparisons on the same queries; a positive interval excludes zero in favor of the left-hand system.

**ANCE.** Base B64 `I+R+D` versus the single B64 IterCQR rewrite gives +0.0458 MRR, 95% CI [+0.0403, +0.0514]. The comparison with B64 IterCQR-G+2S gives +0.0391 MRR, 95% CI [+0.0335, +0.0446]. Against B512 IterCQR-G+2S, the B64 fusion gives +0.0177 MRR, 95% CI [+0.0118, +0.0237]. These are paired comparisons on the same queries; a positive interval excludes zero in favor of the left-hand system.

**BM25.** Base, imitation-control and corrected-label `I+R+D` reach 0.4915, 0.4854 and 0.4937 MRR. Control minus base gives -0.0061 MRR, 95% CI [-0.0097, -0.0024]; treatment minus base gives +0.0022 MRR, 95% CI [-0.0020, +0.0064]; treatment minus control gives +0.0083 MRR, 95% CI [+0.0040, +0.0126]. For standalone `R`, the corresponding base/control/treatment means are 0.4144/0.4014/0.4283.

**ANCE.** Base, imitation-control and corrected-label `I+R+D` reach 0.4233, 0.4211 and 0.4325 MRR. Control minus base gives -0.0022 MRR, 95% CI [-0.0057, +0.0014]; treatment minus base gives +0.0092 MRR, 95% CI [+0.0049, +0.0135]; treatment minus control gives +0.0114 MRR, 95% CI [+0.0069, +0.0157]. For standalone `R`, the corresponding base/control/treatment means are 0.3799/0.3754/0.3894.

## Result summary

These tables read the saved results of this notebook and display them directly, without a separate export. Additional cross-experiment analyses appear after the corresponding supporting scripts have been run; unavailable analyses are identified explicitly.


In [7]:
from experiments.scripts.analysis_results import display_results

display_results(PROJECT_ROOT / "experiments", '07b_qrecc_ood_retrieval')


Additional retrieval comparisons are not available yet: <project>/experiments/results/10_efficiency_analysis/figures/data/results_retrieval_paired_comparisons.csv. Run the README's teacher/figure analysis scripts and rerun this cell. Showing the completed notebook results below.


**QReCC Test / BM25: route means**

,system,budget,arm,n,MRR,nDCG@3,R@10,R@100,R@1000
0,bm25_treatment_e6,64,D,8209,0.479376,0.450424,0.680054,0.913105,0.970397
1,bm25_treatment_e6,64,I,8209,0.439053,0.412792,0.607896,0.810153,0.908771
2,bm25_treatment_e6,64,I+D,8209,0.487714,0.458338,0.702873,0.919767,0.977512
3,bm25_treatment_e6,64,I+R,8209,0.457429,0.432174,0.659385,0.867153,0.950797
4,bm25_treatment_e6,64,I+R+D,8209,0.493690,0.464717,0.709550,0.924389,0.980402
5,bm25_treatment_e6,64,R,8209,0.428304,0.401666,0.610712,0.832555,0.929258
6,bm25_treatment_e6,64,R+D,8209,0.471717,0.440947,0.692258,0.917624,0.977611
7,imitation_control_e6,64,D,8209,0.472498,0.444318,0.660364,0.895697,0.959524
8,imitation_control_e6,64,I,8209,0.439053,0.412792,0.607896,0.810153,0.908771
9,imitation_control_e6,64,I+D,8209,0.489856,0.463143,0.702264,0.913390,0.975528


**QReCC Test / BM25: paired comparisons**

,comparison,metric,left_system,left_budget,left_arm,right_system,right_budget,right_arm,n,delta,ci95_low,ci95_high,seed,replicates
0,pretrained_R_minus_I_B64,MRR,pretrained,64,R,pretrained,64,I,8209,-0.024688,-0.032025,-0.017349,1000,10000
1,pretrained_R_minus_I_B64,nDCG@3,pretrained,64,R,pretrained,64,I,8209,-0.025305,-0.033272,-0.017560,1001,10000
2,pretrained_R_minus_I_B64,R@10,pretrained,64,R,pretrained,64,I,8209,-0.011741,-0.020510,-0.002650,1002,10000
3,pretrained_R_minus_I_B64,R@100,pretrained,64,R,pretrained,64,I,8209,0.007351,-0.000279,0.014946,1003,10000
4,pretrained_R_minus_I_B64,R@1000,pretrained,64,R,pretrained,64,I,8209,0.007102,0.001052,0.013160,1004,10000
5,pretrained_IRD_minus_IterCQR1_B64,MRR,pretrained,64,I+R+D,itercqr,64,IterCQR-1,8209,0.052427,0.046343,0.058463,1005,10000
6,pretrained_IRD_minus_IterCQR1_B64,nDCG@3,pretrained,64,I+R+D,itercqr,64,IterCQR-1,8209,0.049264,0.042748,0.055997,1006,10000
7,pretrained_IRD_minus_IterCQR1_B64,R@10,pretrained,64,I+R+D,itercqr,64,IterCQR-1,8209,0.098672,0.090713,0.106626,1007,10000
8,pretrained_IRD_minus_IterCQR1_B64,R@100,pretrained,64,I+R+D,itercqr,64,IterCQR-1,8209,0.108302,0.101218,0.115377,1008,10000
9,pretrained_IRD_minus_IterCQR1_B64,R@1000,pretrained,64,I+R+D,itercqr,64,IterCQR-1,8209,0.070441,0.065008,0.076072,1009,10000


...

,route,n,mean_tokens,maximum_tokens
0,I512,8209,111.985382,NaN
1,I64,8209,53.206603,NaN
2,R512,8209,42.639786,138.0
3,R64,8209,40.943964,NaN


**input length counts**

,condition,n,queries,percent
0,I64 shorter than I512,8209,5349,65.160190
1,R512 above64 through128,8209,1013,12.340115
2,R512 above128,8209,3,0.036545
3,R512 above64,8209,1016,12.376660


**input length histogram**

,tokens_from,tokens_through,I512,I64,R512
0,1,8,9,9,9
1,9,16,971,971,972
2,17,24,374,374,619
3,25,32,95,95,876
4,33,40,180,180,1338
5,41,48,303,303,1430
6,49,56,580,580,1204
7,57,64,348,5697,745
8,65,72,199,0,424
9,73,80,276,0,272
